# Scriptio Continua Word Segmentation & Sentence Break Pipeline
## Two-Stage Architecture: 4-State + CRF Neural Networks (Layer 1) & Pretrained Transformers (Layer 2)

### 📌 Architecture Overview
This notebook implements an end-to-end, two-stage deep learning pipeline designed to solve **Scriptio Continua** restoration (reconstructing unspaced, unpunctuated text into structured sentences):

```mermaid
flowchart TD
    A[50-Character Scriptio Continua Lines] --> B[Layer 1: Neural Network + CRF]
    B -->|BiLSTM+CRF / BiGRU+CRF / BiRNN+CRF / CNN+CRF| C[Word Boundary Restoration (4-State BIES)]
    C -->|Autoregressive Sliding Window Context: up to 3 prev lines| D[Accumulated Word-Segmented Text]
    D --> E[Layer 2: Pretrained Transformer]
    E -->|DeBERTa-v3 / RoBERTa / BERT / ELECTRA| F[Sentence Break Detection]
    F --> G[Punctuated, Word-Segmented Complete Text]
    G --> H[16 Individual Excel Workbooks + 1 Master Summary Workbook]
```

### 🎯 Key Specifications
1. **Layer 1 (Word Segmentation via 4-State BIES + CRF)**:
   - 4 Neural Network Architectures: **BiLSTM+CRF**, **BiGRU+CRF**, **BiRNN+CRF**, **CNN+CRF**.
   - Character-level 4-state sequence labeling ($B$: Begin, $I$: Inside, $E$: End, $S$: Single).
   - Constrained Linear-Chain CRF layer with invalid transition penalties.
   - **Autoregressive sliding window context**: Each line $t$ is predicted by feeding up to the previous 3 predicted lines as context:
     $$\text{Input}_t = [\hat{y}_{t-3}, \hat{y}_{t-2}, \hat{y}_{t-1}, x_t]$$
2. **Layer 2 (Sentence Break Detection via Pretrained Transformers)**:
   - Top 4 Transformer Architectures:
     1. **DeBERTa-v3** (`microsoft/deberta-v3-small` / `base`)
     2. **RoBERTa** (`roberta-base`)
     3. **BERT** (`bert-base-uncased`)
     4. **ELECTRA** (`google/electra-base-discriminator`)
   - Word boundary classification: Predicts whether a word boundary serves as a sentence termination break (`.`, `?`, `!`).
   - Sentence Break Quality metrics evaluated: Precision, Recall, F1, Accuracy, and Exact Match.
3. **Sequential Two-Stage Training Flow**:
   - First, train Layer 1 Neural Network completely for all 8 epochs (with validation loss & macro F1 tracking and early stopping patience=2).
   - Once Layer 1 NN training is complete, generate autoregressive cascaded predictions (max 3 lines sliding window context) across Train, Validation, and Test splits.
   - Attach the trained NN output to a fresh, non-fine-tuned Transformer.
   - Fine-tune each of the 4 Transformers for 8 epochs on the NN-predicted word boundaries.
   - Predict sentence breaks on test predictions and compute Sentence Break Quality metrics.
   - Generate all 16 individual Excel workbooks + 1 Master Summary Workbook.
4. **Output Workbooks ($4 \times 4 = 16$ pairs + 1 Master Summary)**:
   - Each workbook named `{NN}_{TRANSFORMER}.xlsx` contains:
     - **Sheet 1 (`Train`)**: All IDs (`ART_ID`, `PARA_ID`, `LINE_ID`), Scriptio Continua, Ground Truth, Predicted Output, Predicted BIES, Ground Truth Labels, and all 12 metrics: `BLEU`, `ROUGE_1_F1`, `ROUGE_2_F1`, `ROUGE_L_F1`, `METEOR`, `BERTSCORE_P`, `BERTSCORE_R`, `BERTSCORE_F1`, `F1`, `PRECISION`, `ACCURACY`, `RECALL`.
     - **Sheet 2 (`Validation`)**: Exact same schema on the Validation split.
     - **Sheet 3 (`Test`)**: Exact same schema on the Test split.
     - **Sheet 4 (`Transformer_Sentence_Breaks`)**: `ART_ID`, `PARA_ID`, `LINE_ID`, `SPLIT`, `PREDICTED_LINE` (accumulated NN output), `GROUND_TRUTH_LINE`, and `PREDICTED_LINE_SENT_BREAK` (sentence breaks placed by transformer).
   - **Master Summary Workbook (`All_16_Models_Summary.xlsx`)**: Aggregates all benchmark and sentence break metrics across all 16 pipelines.
5. **Kaggle Dual T4 GPU (T4x2) Accelerator Optimization**:
   - Multi-GPU DataParallel (`torch.nn.DataParallel`).
   - FP16 Mixed Precision (`torch.cuda.amp.autocast`).
   - Pinned memory DataLoader with worker parallelism.


In [1]:
# Install necessary libraries silently in Kaggle
!pip install -q openpyxl xlsxwriter transformers datasets evaluate bert-score accelerate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.3 MB/s eta 0:00:00


In [2]:
import os
import sys
import gc
import re
import math
import time
import random
import warnings
from collections import Counter
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple, Union

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification

warnings.filterwarnings("ignore")

# ── Hardware & Multi-GPU (T4 x 2) Telemetry ────────────────────────────────────
RANDOM_SEED = 42

def set_seed(seed: int = RANDOM_SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(RANDOM_SEED)

num_gpus = torch.cuda.device_count()
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print("=" * 70)
print(f"PyTorch Version   : {torch.__version__}")
print(f"CUDA Available    : {torch.cuda.is_available()}")
print(f"Device Count      : {num_gpus}")
if torch.cuda.is_available():
    for i in range(num_gpus):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}           : {props.name} | VRAM: {props.total_memory / (1024**3):.2f} GB")
    print(f"Primary Device    : {device}")
    print(f"Multi-GPU Mode    : {'Dual T4 DataParallel ENABLED' if num_gpus >= 2 else 'Single GPU'}")
else:
    print("Running on CPU.")
print("=" * 70)


PyTorch Version   : 2.11.0+cu128
CUDA Available    : True
Device Count      : 2
  GPU 0           : Tesla T4 | VRAM: 14.56 GB
  GPU 1           : Tesla T4 | VRAM: 14.56 GB
Primary Device    : cuda:0
Multi-GPU Mode    : Dual T4 DataParallel ENABLED


In [3]:
# ── Configuration DataClass ───────────────────────────────────────────────────
KAGGLE_DATASET_PATH = Path("/kaggle/input/datasets/vallurikeerthiram/scriptio-continua-eng-50-char/2000_articles_50_character_dataset_ART_ID_split_70_15_15.xlsx")
KAGGLE_OUTPUT_DIR = Path("/kaggle/working/")

@dataclass
class PipelineConfig:
    # Dataset and Output paths
    excel_path: Path = KAGGLE_DATASET_PATH
    output_dir: Path = KAGGLE_OUTPUT_DIR
    
    # Layer 1 (Neural Network + CRF) Settings - EXACT MATCH FROM GIT REPO
    window_size: int = 4            # Current line + max 3 previous predicted lines context
    embedding_dim: int = 64         # Exact match from Git
    hidden_dim: int = 128           # Exact match from Git
    cnn_channels: int = 128         # Exact match from Git
    dropout: float = 0.2            # Exact match from Git
    nn_batch_size: int = 64         # Exact match from Git (64)
    nn_epochs: int = 8              # Exact match from Git (8 epochs)
    nn_lr: float = 1e-3             # Exact match from Git (1e-3)
    weight_decay: float = 1e-5      # Exact match from Git (1e-5)
    patience: int = 2               # Exact match from Git (patience 2)
    invalid_penalty_weight: float = 2.0 # Exact match from Git (2.0)
    
    # Layer 2 (Transformer) Settings
    transformer_batch_size: int = 32 # Multi-GPU batch size
    transformer_epochs: int = 8     # Fine-tuning epochs for Transformer (8 epochs)
    transformer_lr: float = 3e-5
    max_seq_length: int = 128
    
    # Evaluation & Telemetry
    use_bertscore: bool = True
    bertscore_batch_size: int = 32
    bertscore_model: str = "roberta-base"
    
    # Execution Scope Controls (Set limit=0 for full dataset)
    train_row_limit: int = 0        # 0 = Full Train (29,160 rows)
    val_row_limit: int = 0          # 0 = Full Validation (6,584 rows)
    test_row_limit: int = 0         # 0 = Full Test (5,868 rows)
    
    # Models to run (Supports subset for quick testing)
    # Configured to ["cnn"] because BiLSTM, BiGRU, and BiRNN (12 models) are already complete!
    # Expected runtime for CNN + 4 Transformers: ~3.9 hours (safely completes well within Kaggle's 12h limit)
    nn_models: List[str] = field(default_factory=lambda: ["cnn"])
    transformer_models: List[str] = field(default_factory=lambda: ["DeBERTa_v3", "RoBERTa", "BERT", "ELECTRA"])
    
    def __post_init__(self):
        # Auto-detect path if running outside standard Kaggle directory
        if not self.excel_path.exists():
            fallbacks = [
                Path("2000_articles_50_character_dataset_ART_ID_split_70_15_15.xlsx"),
                Path("evaluation of sent and line level CRF/2000_articles_50_character_dataset (1).xlsx"),
                Path("c:/Users/keert/Downloads/Phase_2_Final/English/50_Characters/2000_articles_50_character_dataset_ART_ID_split_70_15_15.xlsx")
            ]
            for fb in fallbacks:
                if fb.exists():
                    self.excel_path = fb
                    print(f"Using fallback dataset path: {self.excel_path}")
                    break
        self.output_dir.mkdir(parents=True, exist_ok=True)

config = PipelineConfig()
print(f"Dataset path: {config.excel_path}", flush=True)
print(f"Output directory: {config.output_dir}", flush=True)
print(f"Target Models: {config.nn_models} -> {config.transformer_models}", flush=True)


Dataset path: /kaggle/input/datasets/vallurikeerthiram/scriptio-continua-eng-50-char/2000_articles_50_character_dataset_ART_ID_split_70_15_15.xlsx
Output directory: /kaggle/working
Target Models: ['cnn'] -> ['DeBERTa_v3', 'RoBERTa', 'BERT', 'ELECTRA']


In [4]:
# ── Label Constants & Vocabulary Utilities ────────────────────────────────────
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
PAD_LABEL = -100

LABEL_TO_ID = {"B": 0, "I": 1, "E": 2, "S": 3}
ID_TO_LABEL = {0: "B", 1: "I", 2: "E", 3: "S"}

NN_DISPLAY_NAMES = {
    "bilstm": "BiLSTM+CRF",
    "bigru": "BiGRU+CRF",
    "birnn": "BiRNN+CRF",
    "cnn": "CNN+CRF"
}

TRANSFORMER_REGISTRY = {
    "DeBERTa_v3": "microsoft/deberta-v3-small",
    "RoBERTa": "roberta-base",
    "BERT": "bert-base-uncased",
    "ELECTRA": "google/electra-base-discriminator"
}

def clean_text(value: object) -> str:
    if pd.isna(value):
        return ""
    t = str(value).strip()
    return "" if t.lower() in {"nan", "none", "<na>"} else t

def parse_state4(state_text: object) -> List[str]:
    labels = []
    text = str(state_text).strip()
    tokens = text.split()
    if len(tokens) <= 1:
        tokens = list(text)
    for token in tokens:
        label = token.upper()
        labels.append("I" if label == "M" else label)
    return labels

def reconstruct_from_state4(script_contin: str, labels: Sequence[str]) -> str:
    words: List[str] = []
    current: List[str] = []
    for char, raw_label in zip(script_contin, labels):
        label = "I" if str(raw_label).upper() == "M" else str(raw_label).upper()
        if label == "S":
            if current:
                words.append("".join(current))
                current = []
            words.append(char)
        elif label == "B":
            if current:
                words.append("".join(current))
            current = [char]
        elif label == "I":
            current = current + [char] if current else [char]
        elif label == "E":
            current = current + [char] if current else [char]
            words.append("".join(current))
            current = []
        else:
            current = current + [char] if current else [char]
    if current:
        words.append("".join(current))
    return " ".join(word for word in words if word)

def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    rename_map = {
        "Art Id": "ART_ID", "Art_id": "ART_ID", "ART_ID": "ART_ID",
        "Para Id": "PARA_ID", "para_id": "PARA_ID", "PARA_ID": "PARA_ID",
        "Line Id": "LINE_ID", "Line_id": "LINE_ID", "LINE_ID": "LINE_ID",
        "sent ids": "SENT_IDS", "Sent_id": "SENT_IDS", "SENT_ID": "SENT_IDS",
        "Ori Sent": "SENT_ORI", "Sentence": "SENT_ORI", "SENT_ORI": "SENT_ORI",
        "Scrip Sent": "SCRIPT_CONTIN", "scriptio_continua": "SCRIPT_CONTIN", "SCRIPT_CONTIN": "SCRIPT_CONTIN",
        "4State": "STATE_4", "4state": "STATE_4", "STATE_4": "STATE_4"
    }
    normalized = df.rename(columns={col: rename_map.get(col, col) for col in df.columns}).copy()
    required = ["ART_ID", "PARA_ID", "LINE_ID", "SENT_ORI", "SCRIPT_CONTIN", "STATE_4"]
    for col in required:
        if col not in normalized.columns:
            raise ValueError(f"Missing required column: {col}")
    if "SENT_IDS" not in normalized.columns:
        normalized["SENT_IDS"] = ""
    return normalized

def load_split_data(excel_path: Path, config: PipelineConfig) -> Dict[str, pd.DataFrame]:
    print(f"Loading dataset from: {excel_path} ...", flush=True)
    workbook = pd.ExcelFile(excel_path)
    sheet_lookup = {sheet.lower(): sheet for sheet in workbook.sheet_names}
    aliases = {"train": "train", "validation": "val", "val": "val", "test": "test"}
    split_sheets: Dict[str, str] = {}
    for raw_name, split_name in aliases.items():
        if raw_name in sheet_lookup and split_name not in split_sheets:
            split_sheets[split_name] = sheet_lookup[raw_name]
            
    splits: Dict[str, pd.DataFrame] = {}
    limits = {"train": config.train_row_limit, "val": config.val_row_limit, "test": config.test_row_limit}
    
    for split_name, sheet_name in split_sheets.items():
        df = normalize_columns(workbook.parse(sheet_name, dtype=str))
        for col in ("SENT_ORI", "SCRIPT_CONTIN", "SENT_IDS", "STATE_4"):
            df[col] = df[col].map(clean_text)
        df["PARA_ID"] = pd.to_numeric(df["PARA_ID"], errors="coerce").fillna(0).astype(int)
        df["LINE_ID"] = pd.to_numeric(df["LINE_ID"], errors="coerce").fillna(0).astype(int)
        df["STATE_4_LIST"] = df["STATE_4"].map(parse_state4)
        
        valid_rows = [
            len(row.SCRIPT_CONTIN) > 0 and len(row.STATE_4_LIST) == len(row.SCRIPT_CONTIN)
            for row in df.itertuples(index=False)
        ]
        df = df.loc[valid_rows].copy()
        df["EXPECTED_SPLIT_STATE4"] = [
            reconstruct_from_state4(script, labels)
            for script, labels in zip(df["SCRIPT_CONTIN"], df["STATE_4_LIST"])
        ]
        df = df.sort_values(["ART_ID", "PARA_ID", "LINE_ID"]).reset_index(drop=True)
        if limits.get(split_name, 0) > 0:
            df = df.head(limits[split_name]).copy().reset_index(drop=True)
        splits[split_name] = df
        print(f"  {split_name.upper():<10}: {len(df):,} valid rows across {df['ART_ID'].nunique():,} unique articles.")
        
    return splits

def build_char_vocab(train_df: pd.DataFrame) -> Dict[str, int]:
    counter = Counter()
    for text in train_df["SCRIPT_CONTIN"]:
        counter.update(text)
    counter.update(" ")
    vocab = {PAD_TOKEN: 0, UNK_TOKEN: 1}
    for char in sorted(counter):
        vocab[char] = len(vocab)
    print(f"Character Vocabulary size: {len(vocab)} unique characters.")
    return vocab

def create_embedding_weights(vocab_size: int, embedding_dim: int, seed: int) -> torch.Tensor:
    torch.manual_seed(seed)
    weights = torch.empty(vocab_size, embedding_dim)
    nn.init.xavier_uniform_(weights)
    weights[0].zero_()
    return weights


In [5]:
# ── Sliding Window Autoregressive Dataset ─────────────────────────────────────
def rolling_context_rows(dataframe: pd.DataFrame, window_size: int) -> List[Dict[str, object]]:
    rows: List[Dict[str, object]] = []
    for _, group in dataframe.groupby(["ART_ID", "PARA_ID"], sort=False):
        history: List[str] = []
        for row in group.itertuples(index=False):
            current_script = row.SCRIPT_CONTIN
            context = history[-max(window_size - 1, 0):]
            input_text = " ".join([*context, current_script]) if context else current_script
            current_start = len(input_text) - len(current_script)
            labels = list(row.STATE_4_LIST)
            full_labels = [PAD_LABEL] * current_start + labels
            rows.append({
                "input_text": input_text,
                "current_start": current_start,
                "labels": full_labels,
                "art_id": row.ART_ID,
                "para_id": row.PARA_ID,
                "line_id": row.LINE_ID,
                "sent_ids": row.SENT_IDS,
                "script_contin": current_script,
                "sent_ori": row.SENT_ORI,
                "expected_split": row.EXPECTED_SPLIT_STATE4,
            })
            history.append(row.EXPECTED_SPLIT_STATE4)
    return rows

class LineWiseDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, char_vocab: Dict[str, int], window_size: int):
        self.examples = rolling_context_rows(dataframe, window_size)
        self.char_vocab = char_vocab
        self.label_to_id = LABEL_TO_ID

    def __len__(self) -> int:
        return len(self.examples)

    def __getitem__(self, index: int) -> Dict[str, object]:
        ex = self.examples[index]
        char_ids = [self.char_vocab.get(ch, self.char_vocab[UNK_TOKEN]) for ch in str(ex["input_text"])]
        label_ids = [PAD_LABEL if lbl == PAD_LABEL else self.label_to_id[lbl] for lbl in ex["labels"]]
        return {
            **ex,
            "char_ids": torch.tensor(char_ids, dtype=torch.long),
            "labels_tensor": torch.tensor(label_ids, dtype=torch.long),
        }

def collate_batch(batch: Sequence[Dict[str, object]]) -> Dict[str, object]:
    padded_chars = pad_sequence([item["char_ids"] for item in batch], batch_first=True, padding_value=0)
    padded_labels = pad_sequence([item["labels_tensor"] for item in batch], batch_first=True, padding_value=PAD_LABEL)
    return {
        "char_ids": padded_chars,
        "labels": padded_labels,
        "mask": padded_labels.ne(PAD_LABEL),
        "items": list(batch),
    }

def create_data_loaders(splits: Dict[str, pd.DataFrame], char_vocab: Dict[str, int], config: PipelineConfig) -> Dict[str, DataLoader]:
    loaders = {}
    for split_name, df in splits.items():
        ds = LineWiseDataset(df, char_vocab, config.window_size)
        loaders[split_name] = DataLoader(
            ds,
            batch_size=config.nn_batch_size,
            shuffle=(split_name == "train"),
            num_workers=2 if torch.cuda.is_available() else 0,
            pin_memory=torch.cuda.is_available(),
            collate_fn=collate_batch,
        )
    return loaders


In [6]:
# ── Constrained Linear-Chain CRF Layer ─────────────────────────────────────────
class CRFLayer(nn.Module):
    def __init__(self, num_tags: int, transition_mask: torch.Tensor, start_mask: torch.Tensor, end_mask: torch.Tensor):
        super().__init__()
        self.num_tags = num_tags
        self.start_transitions = nn.Parameter(torch.empty(num_tags))
        self.end_transitions = nn.Parameter(torch.empty(num_tags))
        self.transitions = nn.Parameter(torch.empty(num_tags, num_tags))
        self.register_buffer("transition_mask", transition_mask)
        self.register_buffer("start_mask", start_mask)
        self.register_buffer("end_mask", end_mask)
        self.constraint_value = -1e4
        self.reset_parameters()

    def reset_parameters(self) -> None:
        nn.init.uniform_(self.start_transitions, -0.1, 0.1)
        nn.init.uniform_(self.end_transitions, -0.1, 0.1)
        nn.init.uniform_(self.transitions, -0.1, 0.1)

    def _masked_start_transitions(self) -> torch.Tensor:
        return self.start_transitions.masked_fill(~self.start_mask, self.constraint_value)

    def _masked_end_transitions(self) -> torch.Tensor:
        return self.end_transitions.masked_fill(~self.end_mask, self.constraint_value)

    def _masked_transitions(self) -> torch.Tensor:
        return self.transitions.masked_fill(~self.transition_mask, self.constraint_value)

    def forward(self, emissions: torch.Tensor, tags: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        numerator = self._compute_score(emissions, tags, mask)
        denominator = self._compute_log_partition(emissions, mask)
        return torch.mean(denominator - numerator)

    def _compute_score(self, emissions: torch.Tensor, tags: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        start_tr = self._masked_start_transitions()
        end_tr = self._masked_end_transitions()
        transitions = self._masked_transitions()
        score = start_tr[tags[:, 0]] + emissions[:, 0, :].gather(1, tags[:, 0].unsqueeze(1)).squeeze(1)
        for t in range(1, emissions.size(1)):
            active = mask[:, t]
            prev_tags = tags[:, t - 1]
            curr_tags = tags[:, t]
            emit = emissions[:, t, :].gather(1, curr_tags.unsqueeze(1)).squeeze(1)
            score += (transitions[prev_tags, curr_tags] + emit) * active
        seq_ends = mask.long().sum(dim=1) - 1
        last_tags = tags.gather(1, seq_ends.unsqueeze(1)).squeeze(1)
        score += end_tr[last_tags]
        return score

    def _compute_log_partition(self, emissions: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        score = self._masked_start_transitions() + emissions[:, 0]
        transitions = self._masked_transitions()
        for t in range(1, emissions.size(1)):
            next_score = score.unsqueeze(2) + transitions + emissions[:, t].unsqueeze(1)
            next_score = torch.logsumexp(next_score, dim=1)
            score = torch.where(mask[:, t].unsqueeze(1), next_score, score)
        score = score + self._masked_end_transitions()
        return torch.logsumexp(score, dim=1)

    def decode(self, emissions: torch.Tensor, mask: torch.Tensor) -> List[List[int]]:
        score = self._masked_start_transitions() + emissions[:, 0]
        transitions = self._masked_transitions()
        history: List[torch.Tensor] = []
        for t in range(1, emissions.size(1)):
            next_score = score.unsqueeze(2) + transitions
            best_score, best_path = next_score.max(dim=1)
            best_score = best_score + emissions[:, t]
            score = torch.where(mask[:, t].unsqueeze(1), best_score, score)
            history.append(best_path)
        score = score + self._masked_end_transitions()
        best_last_tags = score.argmax(dim=1)

        paths: List[List[int]] = []
        lengths = mask.long().sum(dim=1).tolist()
        for b_idx, length in enumerate(lengths):
            tag = best_last_tags[b_idx]
            path = [int(tag.item())]
            for hist in reversed(history[: max(length - 1, 0)]):
                tag = hist[b_idx][tag]
                path.append(int(tag.item()))
            paths.append(list(reversed(path)))
        return paths

def build_bies_crf_constraints() -> Tuple[Dict[str, torch.Tensor], Dict[str, torch.Tensor]]:
    num_labels = 4
    transition_mask = torch.zeros(num_labels, num_labels, dtype=torch.bool)
    start_mask = torch.zeros(num_labels, dtype=torch.bool)
    end_mask = torch.zeros(num_labels, dtype=torch.bool)
    valid_next = {"B": ("I", "E"), "I": ("I", "E"), "E": ("B", "S"), "S": ("B", "S")}
    for label in ("B", "S"): start_mask[LABEL_TO_ID[label]] = True
    for label in ("E", "S"): end_mask[LABEL_TO_ID[label]] = True
    for cur, nxt_list in valid_next.items():
        for nxt in nxt_list:
            transition_mask[LABEL_TO_ID[cur], LABEL_TO_ID[nxt]] = True
    return (
        {"transition_mask": transition_mask, "start_mask": start_mask, "end_mask": end_mask},
        {"transition": ~transition_mask, "start": ~start_mask, "end": ~end_mask}
    )

def compute_invalid_bies_penalty(emissions: torch.Tensor, mask: torch.Tensor, invalid_masks: Dict[str, torch.Tensor]) -> torch.Tensor:
    probabilities = torch.softmax(emissions, dim=-1)
    lengths = mask.long().sum(dim=1)
    start_penalty = probabilities[:, 0, :][:, invalid_masks["start"].to(probabilities.device)].sum(dim=1)
    last_positions = (lengths - 1).clamp_min(0)
    last_probabilities = probabilities[torch.arange(probabilities.size(0), device=probabilities.device), last_positions]
    end_penalty = last_probabilities[:, invalid_masks["end"].to(probabilities.device)].sum(dim=1)
    transition_penalty = torch.zeros(probabilities.size(0), device=probabilities.device)
    transition_invalid = invalid_masks["transition"].to(probabilities.device)
    for timestep in range(1, probabilities.size(1)):
        active = mask[:, timestep].float()
        pair_probabilities = probabilities[:, timestep - 1].unsqueeze(2) * probabilities[:, timestep].unsqueeze(1)
        invalid_mass = pair_probabilities.masked_select(transition_invalid.unsqueeze(0)).view(probabilities.size(0), -1).sum(dim=1)
        transition_penalty = transition_penalty + invalid_mass * active
    return ((start_penalty + end_penalty + transition_penalty) / lengths.float().clamp_min(1.0)).mean()

# ── Base Sequence Tagger & Model Variants ──────────────────────────────────────
class SequenceTaggerBase(nn.Module):
    def __init__(self, embedding_weights: torch.Tensor, dropout: float, num_labels: int, crf_kwargs: Dict[str, torch.Tensor], invalid_penalty_masks: Dict[str, torch.Tensor], invalid_penalty_weight: float):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(embedding_weights.clone(), freeze=False, padding_idx=0)
        self.dropout = nn.Dropout(dropout)
        self.num_labels = num_labels
        self.crf = CRFLayer(num_labels, **crf_kwargs)
        self.invalid_masks = invalid_penalty_masks
        self.invalid_weight = invalid_penalty_weight

    def loss(self, inputs: torch.Tensor, tags: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        emissions = self.forward(inputs)
        batch_size = emissions.size(0)
        valid_lens = mask.long().sum(dim=1)
        max_len = valid_lens.max().item()
        packed_emissions = emissions.new_zeros(batch_size, max_len, emissions.size(2))
        packed_tags = tags.new_zeros(batch_size, max_len)
        packed_mask = mask.new_zeros(batch_size, max_len, dtype=torch.bool)
        for i in range(batch_size):
            vlen = valid_lens[i].item()
            if vlen > 0:
                first_true = mask[i].nonzero(as_tuple=True)[0][0].item()
                packed_emissions[i, :vlen] = emissions[i, first_true : first_true + vlen]
                packed_tags[i, :vlen] = tags[i, first_true : first_true + vlen]
                packed_mask[i, :vlen] = True
        safe_tags = packed_tags.masked_fill(~packed_mask, 0)
        loss = self.crf(packed_emissions, safe_tags, packed_mask)
        if self.invalid_masks and self.invalid_weight > 0:
            loss = loss + self.invalid_weight * compute_invalid_bies_penalty(packed_emissions, packed_mask, self.invalid_masks)
        return loss

    def decode(self, inputs: torch.Tensor, mask: torch.Tensor) -> List[List[int]]:
        emissions = self.forward(inputs)
        batch_size = emissions.size(0)
        valid_lens = mask.long().sum(dim=1)
        max_len = valid_lens.max().item()
        packed_emissions = emissions.new_zeros(batch_size, max_len, emissions.size(2))
        packed_mask = mask.new_zeros(batch_size, max_len, dtype=torch.bool)
        for i in range(batch_size):
            vlen = valid_lens[i].item()
            if vlen > 0:
                first_true = mask[i].nonzero(as_tuple=True)[0][0].item()
                packed_emissions[i, :vlen] = emissions[i, first_true : first_true + vlen]
                packed_mask[i, :vlen] = True
        return self.crf.decode(packed_emissions, packed_mask)

class BiLSTMTagger(SequenceTaggerBase):
    def __init__(self, embedding_weights: torch.Tensor, hidden_dim: int, dropout: float, crf_kwargs, inv_masks, inv_w):
        super().__init__(embedding_weights, dropout, 4, crf_kwargs, inv_masks, inv_w)
        self.encoder = nn.LSTM(embedding_weights.size(1), hidden_dim, batch_first=True, bidirectional=True)
        self.classifier = nn.Linear(hidden_dim * 2, 4)
    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        out, _ = self.encoder(self.dropout(self.embedding(inputs)))
        return self.classifier(self.dropout(out))

class BiGRUTagger(SequenceTaggerBase):
    def __init__(self, embedding_weights: torch.Tensor, hidden_dim: int, dropout: float, crf_kwargs, inv_masks, inv_w):
        super().__init__(embedding_weights, dropout, 4, crf_kwargs, inv_masks, inv_w)
        self.encoder = nn.GRU(embedding_weights.size(1), hidden_dim, batch_first=True, bidirectional=True)
        self.classifier = nn.Linear(hidden_dim * 2, 4)
    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        out, _ = self.encoder(self.dropout(self.embedding(inputs)))
        return self.classifier(self.dropout(out))

class BiRNNTagger(SequenceTaggerBase):
    def __init__(self, embedding_weights: torch.Tensor, hidden_dim: int, dropout: float, crf_kwargs, inv_masks, inv_w):
        super().__init__(embedding_weights, dropout, 4, crf_kwargs, inv_masks, inv_w)
        self.encoder = nn.RNN(embedding_weights.size(1), hidden_dim, batch_first=True, bidirectional=True, nonlinearity="tanh")
        self.classifier = nn.Linear(hidden_dim * 2, 4)
    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        out, _ = self.encoder(self.dropout(self.embedding(inputs)))
        return self.classifier(self.dropout(out))

class CNNTagger(SequenceTaggerBase):
    def __init__(self, embedding_weights: torch.Tensor, cnn_channels: int, dropout: float, crf_kwargs, inv_masks, inv_w):
        super().__init__(embedding_weights, dropout, 4, crf_kwargs, inv_masks, inv_w)
        self.conv = nn.Conv1d(embedding_weights.size(1), cnn_channels, kernel_size=3, padding=1)
        self.activation = nn.ReLU()
        self.classifier = nn.Linear(cnn_channels, 4)
    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        x = self.dropout(self.embedding(inputs)).transpose(1, 2)
        x = self.activation(self.conv(x)).transpose(1, 2)
        return self.classifier(self.dropout(x))

def build_nn_model(model_name: str, embedding_weights: torch.Tensor, config: PipelineConfig) -> nn.Module:
    crf_kwargs, inv_masks = build_bies_crf_constraints()
    inv_w = config.invalid_penalty_weight
    if model_name == "bilstm":
        return BiLSTMTagger(embedding_weights, config.hidden_dim, config.dropout, crf_kwargs, inv_masks, inv_w)
    elif model_name == "bigru":
        return BiGRUTagger(embedding_weights, config.hidden_dim, config.dropout, crf_kwargs, inv_masks, inv_w)
    elif model_name == "birnn":
        return BiRNNTagger(embedding_weights, config.hidden_dim, config.dropout, crf_kwargs, inv_masks, inv_w)
    elif model_name == "cnn":
        return CNNTagger(embedding_weights, config.cnn_channels, config.dropout, crf_kwargs, inv_masks, inv_w)
    raise ValueError(f"Unknown NN model: {model_name}")


In [7]:
# ── Layer 2: Transformer Sentence Break Detection ─────────────────────────────
class TransformerSentenceBreakModule:
    def __init__(self, model_key: str, hf_model_name: str, device: torch.device, num_gpus: int):
        self.model_key = model_key
        self.hf_model_name = hf_model_name
        self.device = device
        self.num_gpus = num_gpus
        print(f"Loading Transformer [{model_key}] ({hf_model_name}) ...", flush=True)
        self.tokenizer = AutoTokenizer.from_pretrained(hf_model_name, use_fast=True)
        self.model = AutoModelForTokenClassification.from_pretrained(
            hf_model_name,
            num_labels=2, # Label 0: Continue, Label 1: Sentence Break (.)
            torch_dtype=torch.float32,
            ignore_mismatched_sizes=True
        )
        if num_gpus > 1 and torch.cuda.is_available():
            self.model = nn.DataParallel(self.model)
        self.model.to(device)

    def extract_word_labels(self, ori_sent: str, pred_line: str) -> List[int]:
        pred_words = pred_line.split()
        if not pred_words:
            return []
        ori_words = ori_sent.split()
        def clean_w(w): return re.sub(r"[^\w]", "", w).lower()
        ori_breaks = [bool(re.search(r"[.!?]+$", w)) for w in ori_words]
        labels = [0] * len(pred_words)
        ori_idx = 0
        for pred_idx, pw in enumerate(pred_words):
            cp = clean_w(pw)
            for k in range(ori_idx, min(ori_idx + 6, len(ori_words))):
                co = clean_w(ori_words[k])
                if cp and co and (cp in co or co in cp):
                    if ori_breaks[k]:
                        labels[pred_idx] = 1
                    ori_idx = k + 1
                    break
        return labels

    def prepare_dataset(self, df_predictions: pd.DataFrame, max_len: int = 128) -> List[Dict[str, torch.Tensor]]:
        dataset = []
        for row in df_predictions.itertuples(index=False):
            pred_line = str(getattr(row, "PREDICTED_OUTPUT", "")).strip()
            ori_sent = str(getattr(row, "SENT_ORI", getattr(row, "GROUND_TRUTH", ""))).strip()
            words = pred_line.split()
            if not words:
                continue
            word_labels = self.extract_word_labels(ori_sent, pred_line)
            encoding = self.tokenizer(
                words,
                is_split_into_words=True,
                max_length=max_len,
                padding="max_length",
                truncation=True,
                return_tensors="pt"
            )
            input_ids = encoding["input_ids"].squeeze(0)
            attention_mask = encoding["attention_mask"].squeeze(0)
            word_ids = encoding.word_ids(batch_index=0)
            
            token_labels = []
            prev_word_id = None
            for w_id in word_ids:
                if w_id is None:
                    token_labels.append(PAD_LABEL)
                elif w_id != prev_word_id and w_id < len(word_labels):
                    token_labels.append(word_labels[w_id])
                else:
                    token_labels.append(PAD_LABEL)
                prev_word_id = w_id
                
            dataset.append({
                "input_ids": input_ids,
                "attention_mask": attention_mask,
                "labels": torch.tensor(token_labels, dtype=torch.long)
            })
        return dataset

    def fine_tune(self, train_samples: List[Dict[str, torch.Tensor]], epochs: int, lr: float, batch_size: int):
        if not train_samples:
            return
        self.model.train()
        optimizer = torch.optim.AdamW(self.model.parameters(), lr=lr, weight_decay=1e-2)
        
        # DataLoader
        def collate_fn(batch):
            return {
                "input_ids": torch.stack([b["input_ids"] for b in batch]),
                "attention_mask": torch.stack([b["attention_mask"] for b in batch]),
                "labels": torch.stack([b["labels"] for b in batch])
            }
        loader = DataLoader(train_samples, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
        
        # Class weighting for sparse sentence breaks
        loss_fn = nn.CrossEntropyLoss(weight=torch.tensor([1.0, 3.5], dtype=torch.float32, device=self.device), ignore_index=PAD_LABEL)
        
        for epoch in range(1, epochs + 1):
            total_loss = 0.0
            for batch in loader:
                input_ids = batch["input_ids"].to(self.device)
                att_mask = batch["attention_mask"].to(self.device)
                labels = batch["labels"].to(self.device)
                optimizer.zero_grad()
                outputs = self.model(input_ids=input_ids, attention_mask=att_mask)
                logits = outputs.logits.float()
                loss = loss_fn(logits.view(-1, 2), labels.view(-1))
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                optimizer.step()
                total_loss += loss.item()
            print(f"    [Transformer: {self.model_key}] Epoch {epoch}/{epochs} Loss: {total_loss/max(len(loader),1):.4f}", flush=True)

    def predict_sentence_breaks(self, df_predictions: pd.DataFrame, batch_size: int = 64, max_len: int = 128) -> pd.DataFrame:
        self.model.eval()
        model_to_eval = self.model.module if isinstance(self.model, nn.DataParallel) else self.model
        results = []
        rows = list(df_predictions.itertuples(index=False))
        with torch.no_grad():
            for i in range(0, len(rows), batch_size):
                batch_rows = rows[i : i + batch_size]
                batch_words_list = []
                batch_labels_list = []
                valid_indices = []
                for b_idx, row in enumerate(batch_rows):
                    pred_line = str(getattr(row, "PREDICTED_OUTPUT", "")).strip()
                    ori_sent = str(getattr(row, "SENT_ORI", getattr(row, "GROUND_TRUTH", ""))).strip()
                    words = pred_line.split()
                    if not words:
                        results.append({
                            "ART_ID": row.ART_ID, "PARA_ID": row.PARA_ID, "LINE_ID": row.LINE_ID,
                            "PREDICTED_LINE": pred_line, "GROUND_TRUTH_LINE": ori_sent,
                            "PREDICTED_LINE_SENT_BREAK": pred_line,
                            "TRUE_BREAKS": [], "PRED_BREAKS": []
                        })
                    else:
                        batch_words_list.append(words)
                        batch_labels_list.append(self.extract_word_labels(ori_sent, pred_line))
                        valid_indices.append((row, words, ori_sent))
                
                if not batch_words_list:
                    continue
                    
                encoding = self.tokenizer(
                    batch_words_list, is_split_into_words=True, max_length=max_len,
                    padding=True, truncation=True, return_tensors="pt"
                )
                input_ids = encoding["input_ids"].to(self.device)
                att_mask = encoding["attention_mask"].to(self.device)
                
                outputs = model_to_eval(input_ids=input_ids, attention_mask=att_mask)
                preds = outputs.logits.argmax(dim=-1).cpu().tolist()
                
                for b_idx, (row, words, ori_sent) in enumerate(valid_indices):
                    word_ids = encoding.word_ids(batch_index=b_idx)
                    true_labels = batch_labels_list[b_idx]
                    pred_labels = [0] * len(words)
                    prev_word_id = None
                    for token_idx, w_id in enumerate(word_ids):
                        if w_id is not None and w_id != prev_word_id and w_id < len(words):
                            pred_labels[w_id] = preds[b_idx][token_idx]
                        prev_word_id = w_id
                        
                    reconstructed_words = []
                    for w, is_break in zip(words, pred_labels):
                        if is_break == 1:
                            reconstructed_words.append(w + "." if not w.endswith((".", "!", "?")) else w)
                        else:
                            reconstructed_words.append(w)
                    pred_line_sent_break = " ".join(reconstructed_words)
                    
                    results.append({
                        "ART_ID": row.ART_ID, "PARA_ID": row.PARA_ID, "LINE_ID": row.LINE_ID,
                        "PREDICTED_LINE": str(getattr(row, "PREDICTED_OUTPUT", "")).strip(),
                        "GROUND_TRUTH_LINE": ori_sent,
                        "PREDICTED_LINE_SENT_BREAK": pred_line_sent_break,
                        "TRUE_BREAKS": true_labels, "PRED_BREAKS": pred_labels
                    })
        return pd.DataFrame(results)


In [8]:
# ── Exact Benchmark Metrics Engine ───────────────────────────────────────────
def ngrams(tokens: Sequence[str], n: int) -> Counter:
    return Counter(tuple(tokens[i : i + n]) for i in range(len(tokens) - n + 1)) if len(tokens) >= n else Counter()

def lcs_length(a: Sequence[str], b: Sequence[str]) -> int:
    if not a or not b: return 0
    dp = [0] * (len(b) + 1)
    for x in a:
        prev = 0
        for j, y in enumerate(b):
            temp = dp[j + 1]
            dp[j + 1] = prev + 1 if x == y else max(dp[j + 1], prev)
            prev = temp
    return dp[-1]

def calc_line_bleu(ref: str, cand: str, max_n: int = 4) -> float:
    ref_tokens = str(ref).split()
    cand_tokens = str(cand).split()
    cand_len = len(cand_tokens)
    ref_len = len(ref_tokens)
    if cand_len == 0 or ref_len == 0:
        return 0.0
    precisions = []
    for n in range(1, max_n + 1):
        ref_ng = ngrams(ref_tokens, n)
        cand_ng = ngrams(cand_tokens, n)
        cand_total = sum(cand_ng.values())
        if cand_total == 0:
            p = 1.0 / (cand_len + 1)
        else:
            overlap = sum(min(count, ref_ng[gram]) for gram, count in cand_ng.items())
            p = (overlap + 0.1) / (cand_total + 0.1)
        precisions.append(p)
    geo_mean = math.exp(sum(math.log(p) for p in precisions) / max_n)
    bp = 1.0 if cand_len > ref_len else math.exp(1 - (ref_len / cand_len))
    return float(geo_mean * bp)

def calc_line_rouges(ref: str, cand: str) -> Tuple[float, float, float]:
    ref_tokens = str(ref).split()
    cand_tokens = str(cand).split()
    if not ref_tokens or not cand_tokens:
        return 0.0, 0.0, 0.0
    r1_cand = Counter(cand_tokens)
    r1_ref = Counter(ref_tokens)
    overlap1 = sum(min(c, r1_ref[w]) for w, c in r1_cand.items())
    p1 = overlap1 / len(cand_tokens)
    rc1 = overlap1 / len(ref_tokens)
    f1_1 = (2 * p1 * rc1 / (p1 + rc1)) if (p1 + rc1) > 0 else 0.0

    if len(cand_tokens) >= 2 and len(ref_tokens) >= 2:
        r2_cand = ngrams(cand_tokens, 2)
        r2_ref = ngrams(ref_tokens, 2)
        overlap2 = sum(min(c, r2_ref[g]) for g, c in r2_cand.items())
        p2 = overlap2 / (len(cand_tokens) - 1)
        rc2 = overlap2 / (len(ref_tokens) - 1)
        f1_2 = (2 * p2 * rc2 / (p2 + rc2)) if (p2 + rc2) > 0 else 0.0
    else:
        f1_2 = 0.0

    lcs = lcs_length(ref_tokens, cand_tokens)
    pl = lcs / len(cand_tokens)
    rcl = lcs / len(ref_tokens)
    f1_l = (2 * pl * rcl / (pl + rcl)) if (pl + rcl) > 0 else 0.0
    return float(f1_1), float(f1_2), float(f1_l)

def calc_line_meteor(ref: str, cand: str) -> float:
    ref_tokens = str(ref).split()
    cand_tokens = str(cand).split()
    if not ref_tokens or not cand_tokens:
        return 0.0
    matches = 0
    ref_matched = [False] * len(ref_tokens)
    cand_matched = [False] * len(cand_tokens)
    for i, c in enumerate(cand_tokens):
        for j, r in enumerate(ref_tokens):
            if not ref_matched[j] and c == r:
                ref_matched[j] = True
                cand_matched[i] = True
                matches += 1
                break
    if matches == 0:
        return 0.0
    p = matches / len(cand_tokens)
    r = matches / len(ref_tokens)
    fmean = (10 * p * r) / (9 * p + r) if (9 * p + r) > 0 else 0.0
    chunks = 0
    in_chunk = False
    for m in cand_matched:
        if m and not in_chunk:
            chunks += 1
            in_chunk = True
        elif not m:
            in_chunk = False
    frag = chunks / matches if matches else 0.0
    penalty = 0.5 * (frag ** 3)
    return float(fmean * (1.0 - penalty))

def calc_state4_classification_metrics(true_str: str, pred_str: str) -> Tuple[float, float, float, float]:
    true = str(true_str).split()
    pred = str(pred_str).split()
    n = min(len(true), len(pred))
    if n == 0:
        return 0.0, 0.0, 0.0, 0.0
    labels = ["B", "I", "E", "S"]
    correct = sum(1 for t, p in zip(true[:n], pred[:n]) if t == p)
    acc = correct / n
    p_list, r_list, f1_list = [], [], []
    for lbl in labels:
        tp = sum(1 for t, p in zip(true[:n], pred[:n]) if t == lbl and p == lbl)
        fp = sum(1 for t, p in zip(true[:n], pred[:n]) if t != lbl and p == lbl)
        fn = sum(1 for t, p in zip(true[:n], pred[:n]) if t == lbl and p != lbl)
        if (tp + fn) > 0 or (tp + fp) > 0:
            p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
            r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            f = (2 * p * r / (p + r)) if (p + r) > 0 else 0.0
            p_list.append(p)
            r_list.append(r)
            f1_list.append(f)
    prec = sum(p_list) / len(p_list) if p_list else 1.0
    rec = sum(r_list) / len(r_list) if r_list else 1.0
    f1 = sum(f1_list) / len(f1_list) if f1_list else 1.0
    return float(acc), float(prec), float(rec), float(f1)

# ── Sentence Break Quality Metrics ────────────────────────────────────────────
def compute_sentence_break_quality_metrics(df_sent_breaks: pd.DataFrame) -> Dict[str, float]:
    total_tp = 0
    total_fp = 0
    total_fn = 0
    total_tokens = 0
    total_correct_tokens = 0
    exact_matches = 0
    total_rows = len(df_sent_breaks)
    
    for row in df_sent_breaks.itertuples(index=False):
        tb = list(row.TRUE_BREAKS)
        pb = list(row.PRED_BREAKS)
        n = min(len(tb), len(pb))
        if n == 0:
            continue
        row_correct = True
        for t, p in zip(tb[:n], pb[:n]):
            total_tokens += 1
            if t == 1 and p == 1:
                total_tp += 1
            elif t == 0 and p == 1:
                total_fp += 1
                row_correct = False
            elif t == 1 and p == 0:
                total_fn += 1
                row_correct = False
            if t == p:
                total_correct_tokens += 1
        if row_correct and tb[:n] == pb[:n]:
            exact_matches += 1
            
    p = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    r = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
    f1 = (2 * p * r / (p + r)) if (p + r) > 0 else 0.0
    acc = total_correct_tokens / total_tokens if total_tokens > 0 else 0.0
    em = exact_matches / total_rows if total_rows > 0 else 0.0
    
    return {
        "SENT_BREAK_PRECISION": float(p),
        "SENT_BREAK_RECALL": float(r),
        "SENT_BREAK_F1": float(f1),
        "SENT_BREAK_ACCURACY": float(acc),
        "SENT_BREAK_EXACT_MATCH": float(em)
    }

def compute_bert_scores(candidates: List[str], references: List[str], batch_size: int = 32) -> Tuple[List[float], List[float], List[float]]:
    try:
        from bert_score import score as bert_score_fn
        P, R, F1 = bert_score_fn(candidates, references, model_type="roberta-base", lang="en", verbose=False, batch_size=batch_size)
        return P.tolist(), R.tolist(), F1.tolist()
    except Exception as exc:
        warnings.warn(f"BERTScore computation fallback: {exc}")
        n = len(candidates)
        return [float("nan")] * n, [float("nan")] * n, [float("nan")] * n


In [9]:
# ── Autoregressive Cascaded Prediction Engine (Layer 1) ───────────────────────
def predict_linewise_cascade(
    model: nn.Module,
    dataframe: pd.DataFrame,
    char_vocab: Dict[str, int],
    device: torch.device,
    window_size: int,
) -> pd.DataFrame:
    model.eval()
    rows: List[Dict[str, object]] = []
    with torch.no_grad():
        for _, group in dataframe.groupby(["ART_ID", "PARA_ID"], sort=False):
            predicted_history: List[str] = []
            for row in group.itertuples(index=False):
                context = predicted_history[-max(window_size - 1, 0):]
                input_text = " ".join([*context, row.SCRIPT_CONTIN]) if context else row.SCRIPT_CONTIN
                current_start = len(input_text) - len(row.SCRIPT_CONTIN)
                char_ids = [char_vocab.get(ch, char_vocab[UNK_TOKEN]) for ch in input_text]
                inputs = torch.tensor([char_ids], dtype=torch.long, device=device)
                mask = torch.ones_like(inputs, dtype=torch.bool, device=device)
                
                # Model decode
                decoded = model.decode(inputs, mask)[0]
                line_decoded = decoded[current_start : current_start + len(row.SCRIPT_CONTIN)]
                pred_labels = [ID_TO_LABEL[tag] for tag in line_decoded]
                true_labels = list(row.STATE_4_LIST)
                predicted_output = reconstruct_from_state4(row.SCRIPT_CONTIN, pred_labels)
                
                rows.append({
                    "ART_ID": row.ART_ID,
                    "PARA_ID": row.PARA_ID,
                    "LINE_ID": row.LINE_ID,
                    "SENT_IDS": row.SENT_IDS,
                    "INPUT_TEXT": input_text,
                    "SCRIPTIO_CONTINUA": row.SCRIPT_CONTIN,
                    "SENT_ORI": row.SENT_ORI,
                    "GROUND_TRUTH": row.EXPECTED_SPLIT_STATE4,
                    "PREDICTED_OUTPUT": predicted_output,
                    "GROUND_TRUTH_LABELS": " ".join(str(x) for x in true_labels),
                    "PREDICTED_BIES": " ".join(str(x) for x in pred_labels),
                })
                predicted_history.append(predicted_output)
    return pd.DataFrame(rows)

def train_nn_one_epoch(model: nn.Module, loader: DataLoader, optimizer: torch.optim.Optimizer, device: torch.device) -> float:
    model.train()
    total_loss = 0.0
    for batch in loader:
        inputs = batch["char_ids"].to(device)
        labels = batch["labels"].to(device)
        mask = batch["mask"].to(device)
        optimizer.zero_grad()
        loss = model.loss(inputs, labels, mask)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / max(len(loader), 1)

def evaluate_nn_val(model: nn.Module, loader: DataLoader, device: torch.device) -> Tuple[float, float]:
    model.eval()
    total_loss = 0.0
    all_true: List[int] = []
    all_pred: List[int] = []
    with torch.no_grad():
        for batch in loader:
            inputs = batch["char_ids"].to(device)
            labels = batch["labels"].to(device)
            mask = batch["mask"].to(device)
            loss = model.loss(inputs, labels, mask)
            total_loss += loss.item()
            decoded_ids = model.decode(inputs, mask)
            for idx, pred_seq in enumerate(decoded_ids):
                true_seq = labels[idx][mask[idx]].tolist()
                all_pred.extend(pred_seq)
                all_true.extend(true_seq)
                
    avg_loss = total_loss / max(len(loader), 1)
    matrix = np.zeros((4, 4), dtype=np.int64)
    for t, p in zip(all_true, all_pred):
        if 0 <= t < 4 and 0 <= p < 4:
            matrix[t, p] += 1
            
    f1_list = []
    for idx in range(4):
        tp = matrix[idx, idx]
        fp = matrix[:, idx].sum() - tp
        fn = matrix[idx, :].sum() - tp
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = (2 * prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0
        f1_list.append(f1)
    macro_f1 = float(np.mean(f1_list)) if f1_list else 0.0
    return avg_loss, macro_f1

def fit_neural_network(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
    epochs: int,
    patience: int,
    model_name: str,
) -> nn.Module:
    best_state = None
    best_val_f1 = -1.0
    patience_counter = 0
    print(f"\n--- Training Layer 1 NN [{model_name}] for {epochs} epochs (Early stopping patience={patience}) ---", flush=True)
    for epoch in range(1, epochs + 1):
        train_loss = train_nn_one_epoch(model, train_loader, optimizer, device)
        val_loss, val_f1 = evaluate_nn_val(model, val_loader, device)
        print(
            f"  [NN: {model_name}] Epoch {epoch}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val F1: {val_f1:.4f}",
            flush=True,
        )
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"  [NN: {model_name}] Early stopping triggered at epoch {epoch} (best Val F1: {best_val_f1:.4f})", flush=True)
                break
    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"  [NN: {model_name}] Restored best checkpoint with Val F1: {best_val_f1:.4f}", flush=True)
    return model


In [10]:
# ── Enrich Prediction DataFrames with Full Metrics ─────────────────────────────
def enrich_split_with_metrics(df_split: pd.DataFrame, use_bertscore: bool = True, bertscore_batch_size: int = 32) -> pd.DataFrame:
    df = df_split.copy()
    bleus, r1s, r2s, rls, meteors = [], [], [], [], []
    accs, precs, recs, f1s = [], [], [], []
    
    for row in df.itertuples(index=False):
        ref = str(row.GROUND_TRUTH)
        cand = str(row.PREDICTED_OUTPUT)
        bleus.append(calc_line_bleu(ref, cand))
        r1, r2, rl = calc_line_rouges(ref, cand)
        r1s.append(r1); r2s.append(r2); rls.append(rl)
        meteors.append(calc_line_meteor(ref, cand))
        
        acc, prec, rec, f1 = calc_state4_classification_metrics(row.GROUND_TRUTH_LABELS, row.PREDICTED_BIES)
        accs.append(acc); precs.append(prec); recs.append(rec); f1s.append(f1)
        
    df["BLEU"] = bleus
    df["ROUGE_1_F1"] = r1s
    df["ROUGE_2_F1"] = r2s
    df["ROUGE_L_F1"] = rls
    df["METEOR"] = meteors
    
    if use_bertscore and len(df) > 0:
        bp, br, bf1 = compute_bert_scores(df["PREDICTED_OUTPUT"].tolist(), df["GROUND_TRUTH"].tolist(), bertscore_batch_size)
        df["BERTSCORE_P"] = bp
        df["BERTSCORE_R"] = br
        df["BERTSCORE_F1"] = bf1
    else:
        df["BERTSCORE_P"] = [float("nan")] * len(df)
        df["BERTSCORE_R"] = [float("nan")] * len(df)
        df["BERTSCORE_F1"] = [float("nan")] * len(df)
        
    df["F1"] = f1s
    df["PRECISION"] = precs
    df["ACCURACY"] = accs
    df["RECALL"] = recs
    
    # Reorder columns as requested
    ordered_cols = [
        "ART_ID", "PARA_ID", "LINE_ID", "SCRIPTIO_CONTINUA", "GROUND_TRUTH",
        "PREDICTED_OUTPUT", "PREDICTED_BIES", "GROUND_TRUTH_LABELS",
        "BLEU", "ROUGE_1_F1", "ROUGE_2_F1", "ROUGE_L_F1", "METEOR",
        "BERTSCORE_P", "BERTSCORE_R", "BERTSCORE_F1",
        "F1", "PRECISION", "ACCURACY", "RECALL"
    ]
    return df[ordered_cols]


In [11]:
# ── Excel Workbook Writer for 16 Pipelines + Summary ──────────────────────────
def write_pipeline_workbook(
    file_path: Path,
    df_train_m: pd.DataFrame,
    df_val_m: pd.DataFrame,
    df_test_m: pd.DataFrame,
    df_sentence_breaks: pd.DataFrame
):
    print(f"Writing Excel Workbook: {file_path.name} ...", flush=True)
    with pd.ExcelWriter(file_path, engine="xlsxwriter") as writer:
        wb = writer.book
        hdr_fmt = wb.add_format({"bold": True, "font_name": "Calibri", "font_size": 11, "bg_color": "#1F497D", "font_color": "white", "border": 1, "align": "center", "valign": "vcenter"})
        num_fmt = wb.add_format({"num_format": "0.0000", "border": 1, "valign": "vcenter"})
        int_fmt = wb.add_format({"num_format": "#,##0", "border": 1, "valign": "vcenter"})
        
        # Helper to write detailed split sheet
        def write_split_sheet(df: pd.DataFrame, sheet_name: str):
            df.to_excel(writer, sheet_name=sheet_name, index=False)
            ws = writer.sheets[sheet_name]
            ws.freeze_panes(1, 3)
            ws.autofilter(0, 0, len(df), len(df.columns) - 1)
            for col_idx, col in enumerate(df.columns):
                ws.write(0, col_idx, col, hdr_fmt)
            ws.set_column("A:A", 12)       # ART_ID
            ws.set_column("B:C", 10)       # PARA_ID, LINE_ID
            ws.set_column("D:D", 45)       # SCRIPTIO_CONTINUA
            ws.set_column("E:E", 45)       # GROUND_TRUTH
            ws.set_column("F:F", 45)       # PREDICTED_OUTPUT
            ws.set_column("G:H", 40)       # LABELS
            ws.set_column("I:T", 14, num_fmt) # METRICS
            
        # Sheet 1: Train, Sheet 2: Validation, Sheet 3: Test
        write_split_sheet(df_train_m, "Train")
        write_split_sheet(df_val_m, "Validation")
        write_split_sheet(df_test_m, "Test")
        
        # Sheet 4: Transformer Sentence Breaks
        sheet_sb = "Transformer_Sentence_Breaks"
        sb_cols = ["ART_ID", "PARA_ID", "LINE_ID", "PREDICTED_LINE", "GROUND_TRUTH_LINE", "PREDICTED_LINE_SENT_BREAK"]
        df_sb_sub = df_sentence_breaks[sb_cols]
        df_sb_sub.to_excel(writer, sheet_name=sheet_sb, index=False)
        ws_sb = writer.sheets[sheet_sb]
        ws_sb.freeze_panes(1, 3)
        ws_sb.autofilter(0, 0, len(df_sb_sub), len(df_sb_sub.columns) - 1)
        for col_idx, col in enumerate(df_sb_sub.columns):
            ws_sb.write(0, col_idx, col, hdr_fmt)
        ws_sb.set_column("A:A", 12)
        ws_sb.set_column("B:C", 10)
        ws_sb.set_column("D:D", 50)
        ws_sb.set_column("E:E", 50)
        ws_sb.set_column("F:F", 50)

def write_master_summary_workbook(file_path: Path, summary_records: List[Dict[str, object]]):
    print(f"\nWriting Master Summary Workbook: {file_path.name} ...", flush=True)
    df_summary = pd.DataFrame(summary_records)
    with pd.ExcelWriter(file_path, engine="xlsxwriter") as writer:
        wb = writer.book
        hdr_fmt = wb.add_format({"bold": True, "font_name": "Calibri", "font_size": 11, "bg_color": "#1F497D", "font_color": "white", "border": 1, "align": "center", "valign": "vcenter"})
        num_fmt = wb.add_format({"num_format": "0.0000", "border": 1, "valign": "vcenter"})
        
        sheet_name = "Overall_Summary"
        df_summary.to_excel(writer, sheet_name=sheet_name, index=False)
        ws = writer.sheets[sheet_name]
        ws.freeze_panes(1, 4)
        ws.autofilter(0, 0, len(df_summary), len(df_summary.columns) - 1)
        for col_idx, col in enumerate(df_summary.columns):
            ws.write(0, col_idx, col, hdr_fmt)
        ws.set_column("A:A", 32)       # WORKBOOK_NAME
        ws.set_column("B:C", 18)       # NN_LAYER1, TRANSFORMER_LAYER2
        ws.set_column("D:D", 14)       # SPLIT
        ws.set_column("E:U", 15, num_fmt)


In [12]:
# ── Precomputed Benchmark Metrics for Completed Models (BiLSTM, BiGRU, BiRNN) ──
import json

PRECOMPUTED_12_MODELS_SUMMARY = json.loads(r'''[
  {
    "WORKBOOK_NAME": "BiGRU_CRF_BERT.xlsx",
    "NN_LAYER1": "BiGRU+CRF",
    "TRANSFORMER_LAYER2": "BERT",
    "SPLIT": "TRAIN",
    "BLEU": 0.851376327234309,
    "ROUGE_1_F1": 0.9370009501999209,
    "ROUGE_2_F1": 0.8778791042689829,
    "ROUGE_L_F1": 0.9034849531307106,
    "METEOR": 0.9172449152890737,
    "BERTSCORE_P": 0.9870572349808879,
    "BERTSCORE_R": 0.9892719987775384,
    "BERTSCORE_F1": 0.9881470766469844,
    "F1": 0.9711799569550172,
    "PRECISION": 0.9731246464966876,
    "ACCURACY": 0.9807946980630747,
    "RECALL": 0.9716604516487528,
    "SENT_BREAK_PRECISION": null,
    "SENT_BREAK_RECALL": null,
    "SENT_BREAK_F1": null,
    "SENT_BREAK_ACCURACY": null,
    "SENT_BREAK_EXACT_MATCH": null
  },
  {
    "WORKBOOK_NAME": "BiGRU_CRF_BERT.xlsx",
    "NN_LAYER1": "BiGRU+CRF",
    "TRANSFORMER_LAYER2": "BERT",
    "SPLIT": "VALIDATION",
    "BLEU": 0.8018229490654185,
    "ROUGE_1_F1": 0.9119110615467887,
    "ROUGE_2_F1": 0.8370013256012967,
    "ROUGE_L_F1": 0.8652870868576484,
    "METEOR": 0.8894267343755438,
    "BERTSCORE_P": 0.9809368257280688,
    "BERTSCORE_R": 0.9843920004505433,
    "BERTSCORE_F1": 0.9826354143251625,
    "F1": 0.9582471992199949,
    "PRECISION": 0.9603424792846333,
    "ACCURACY": 0.9719562855709735,
    "RECALL": 0.9591541470093101,
    "SENT_BREAK_PRECISION": null,
    "SENT_BREAK_RECALL": null,
    "SENT_BREAK_F1": null,
    "SENT_BREAK_ACCURACY": null,
    "SENT_BREAK_EXACT_MATCH": null
  },
  {
    "WORKBOOK_NAME": "BiGRU_CRF_BERT.xlsx",
    "NN_LAYER1": "BiGRU+CRF",
    "TRANSFORMER_LAYER2": "BERT",
    "SPLIT": "TEST",
    "BLEU": 0.8194273778778034,
    "ROUGE_1_F1": 0.9225942801386773,
    "ROUGE_2_F1": 0.852322055620903,
    "ROUGE_L_F1": 0.8801978505678816,
    "METEOR": 0.9008846235919223,
    "BERTSCORE_P": 0.9834074480317816,
    "BERTSCORE_R": 0.9866347660985049,
    "BERTSCORE_F1": 0.9849926919654298,
    "F1": 0.9644028374463933,
    "PRECISION": 0.9661849295229392,
    "ACCURACY": 0.9758535339627262,
    "RECALL": 0.9653354291712203,
    "SENT_BREAK_PRECISION": 0.6960886119764624,
    "SENT_BREAK_RECALL": 0.6793918918918919,
    "SENT_BREAK_F1": 0.6876389126346385,
    "SENT_BREAK_ACCURACY": 0.9584555563135275,
    "SENT_BREAK_EXACT_MATCH": 0.7462076018408045
  },
  {
    "WORKBOOK_NAME": "BiGRU_CRF_DeBERTa_v3.xlsx",
    "NN_LAYER1": "BiGRU+CRF",
    "TRANSFORMER_LAYER2": "DeBERTa_v3",
    "SPLIT": "TRAIN",
    "BLEU": 0.851376327234309,
    "ROUGE_1_F1": 0.9370009501999209,
    "ROUGE_2_F1": 0.8778791042689829,
    "ROUGE_L_F1": 0.9034849531307106,
    "METEOR": 0.9172449152890737,
    "BERTSCORE_P": 0.9870572349808879,
    "BERTSCORE_R": 0.9892719987775384,
    "BERTSCORE_F1": 0.9881470766469844,
    "F1": 0.9711799569550172,
    "PRECISION": 0.9731246464966876,
    "ACCURACY": 0.9807946980630747,
    "RECALL": 0.9716604516487528,
    "SENT_BREAK_PRECISION": null,
    "SENT_BREAK_RECALL": null,
    "SENT_BREAK_F1": null,
    "SENT_BREAK_ACCURACY": null,
    "SENT_BREAK_EXACT_MATCH": null
  },
  {
    "WORKBOOK_NAME": "BiGRU_CRF_DeBERTa_v3.xlsx",
    "NN_LAYER1": "BiGRU+CRF",
    "TRANSFORMER_LAYER2": "DeBERTa_v3",
    "SPLIT": "VALIDATION",
    "BLEU": 0.8018229490654185,
    "ROUGE_1_F1": 0.9119110615467887,
    "ROUGE_2_F1": 0.8370013256012967,
    "ROUGE_L_F1": 0.8652870868576484,
    "METEOR": 0.8894267343755438,
    "BERTSCORE_P": 0.9809368257280688,
    "BERTSCORE_R": 0.9843920004505433,
    "BERTSCORE_F1": 0.9826354143251625,
    "F1": 0.9582471992199949,
    "PRECISION": 0.9603424792846333,
    "ACCURACY": 0.9719562855709735,
    "RECALL": 0.9591541470093101,
    "SENT_BREAK_PRECISION": null,
    "SENT_BREAK_RECALL": null,
    "SENT_BREAK_F1": null,
    "SENT_BREAK_ACCURACY": null,
    "SENT_BREAK_EXACT_MATCH": null
  },
  {
    "WORKBOOK_NAME": "BiGRU_CRF_DeBERTa_v3.xlsx",
    "NN_LAYER1": "BiGRU+CRF",
    "TRANSFORMER_LAYER2": "DeBERTa_v3",
    "SPLIT": "TEST",
    "BLEU": 0.8194273778778034,
    "ROUGE_1_F1": 0.9225942801386773,
    "ROUGE_2_F1": 0.852322055620903,
    "ROUGE_L_F1": 0.8801978505678816,
    "METEOR": 0.9008846235919223,
    "BERTSCORE_P": 0.9834074480317816,
    "BERTSCORE_R": 0.9866347660985049,
    "BERTSCORE_F1": 0.9849926919654298,
    "F1": 0.9644028374463933,
    "PRECISION": 0.9661849295229392,
    "ACCURACY": 0.9758535339627262,
    "RECALL": 0.9653354291712203,
    "SENT_BREAK_PRECISION": 0.7813141683778234,
    "SENT_BREAK_RECALL": 0.7712837837837838,
    "SENT_BREAK_F1": 0.7762665759945596,
    "SENT_BREAK_ACCURACY": 0.9700752666166406,
    "SENT_BREAK_EXACT_MATCH": 0.8200102266916652
  },
  {
    "WORKBOOK_NAME": "BiGRU_CRF_ELECTRA.xlsx",
    "NN_LAYER1": "BiGRU+CRF",
    "TRANSFORMER_LAYER2": "ELECTRA",
    "SPLIT": "TRAIN",
    "BLEU": 0.851376327234309,
    "ROUGE_1_F1": 0.9370009501999209,
    "ROUGE_2_F1": 0.8778791042689829,
    "ROUGE_L_F1": 0.9034849531307106,
    "METEOR": 0.9172449152890737,
    "BERTSCORE_P": 0.9870572349808879,
    "BERTSCORE_R": 0.9892719987775384,
    "BERTSCORE_F1": 0.9881470766469844,
    "F1": 0.9711799569550172,
    "PRECISION": 0.9731246464966876,
    "ACCURACY": 0.9807946980630747,
    "RECALL": 0.9716604516487528,
    "SENT_BREAK_PRECISION": null,
    "SENT_BREAK_RECALL": null,
    "SENT_BREAK_F1": null,
    "SENT_BREAK_ACCURACY": null,
    "SENT_BREAK_EXACT_MATCH": null
  },
  {
    "WORKBOOK_NAME": "BiGRU_CRF_ELECTRA.xlsx",
    "NN_LAYER1": "BiGRU+CRF",
    "TRANSFORMER_LAYER2": "ELECTRA",
    "SPLIT": "VALIDATION",
    "BLEU": 0.8018229490654185,
    "ROUGE_1_F1": 0.9119110615467887,
    "ROUGE_2_F1": 0.8370013256012967,
    "ROUGE_L_F1": 0.8652870868576484,
    "METEOR": 0.8894267343755438,
    "BERTSCORE_P": 0.9809368257280688,
    "BERTSCORE_R": 0.9843920004505433,
    "BERTSCORE_F1": 0.9826354143251625,
    "F1": 0.9582471992199949,
    "PRECISION": 0.9603424792846333,
    "ACCURACY": 0.9719562855709735,
    "RECALL": 0.9591541470093101,
    "SENT_BREAK_PRECISION": null,
    "SENT_BREAK_RECALL": null,
    "SENT_BREAK_F1": null,
    "SENT_BREAK_ACCURACY": null,
    "SENT_BREAK_EXACT_MATCH": null
  },
  {
    "WORKBOOK_NAME": "BiGRU_CRF_ELECTRA.xlsx",
    "NN_LAYER1": "BiGRU+CRF",
    "TRANSFORMER_LAYER2": "ELECTRA",
    "SPLIT": "TEST",
    "BLEU": 0.8194273778778034,
    "ROUGE_1_F1": 0.9225942801386773,
    "ROUGE_2_F1": 0.852322055620903,
    "ROUGE_L_F1": 0.8801978505678816,
    "METEOR": 0.9008846235919223,
    "BERTSCORE_P": 0.9834074480317816,
    "BERTSCORE_R": 0.9866347660985049,
    "BERTSCORE_F1": 0.9849926919654298,
    "F1": 0.9644028374463933,
    "PRECISION": 0.9661849295229392,
    "ACCURACY": 0.9758535339627262,
    "RECALL": 0.9653354291712203,
    "SENT_BREAK_PRECISION": 0.683415435139573,
    "SENT_BREAK_RECALL": 0.7030405405405405,
    "SENT_BREAK_F1": 0.6930890924229808,
    "SENT_BREAK_ACCURACY": 0.9580917297678332,
    "SENT_BREAK_EXACT_MATCH": 0.7451849326742799
  },
  {
    "WORKBOOK_NAME": "BiGRU_CRF_RoBERTa.xlsx",
    "NN_LAYER1": "BiGRU+CRF",
    "TRANSFORMER_LAYER2": "RoBERTa",
    "SPLIT": "TRAIN",
    "BLEU": 0.851376327234309,
    "ROUGE_1_F1": 0.9370009501999209,
    "ROUGE_2_F1": 0.8778791042689829,
    "ROUGE_L_F1": 0.9034849531307106,
    "METEOR": 0.9172449152890737,
    "BERTSCORE_P": 0.9870572349808879,
    "BERTSCORE_R": 0.9892719987775384,
    "BERTSCORE_F1": 0.9881470766469844,
    "F1": 0.9711799569550172,
    "PRECISION": 0.9731246464966876,
    "ACCURACY": 0.9807946980630747,
    "RECALL": 0.9716604516487528,
    "SENT_BREAK_PRECISION": null,
    "SENT_BREAK_RECALL": null,
    "SENT_BREAK_F1": null,
    "SENT_BREAK_ACCURACY": null,
    "SENT_BREAK_EXACT_MATCH": null
  },
  {
    "WORKBOOK_NAME": "BiGRU_CRF_RoBERTa.xlsx",
    "NN_LAYER1": "BiGRU+CRF",
    "TRANSFORMER_LAYER2": "RoBERTa",
    "SPLIT": "VALIDATION",
    "BLEU": 0.8018229490654185,
    "ROUGE_1_F1": 0.9119110615467887,
    "ROUGE_2_F1": 0.8370013256012967,
    "ROUGE_L_F1": 0.8652870868576484,
    "METEOR": 0.8894267343755438,
    "BERTSCORE_P": 0.9809368257280688,
    "BERTSCORE_R": 0.9843920004505433,
    "BERTSCORE_F1": 0.9826354143251625,
    "F1": 0.9582471992199949,
    "PRECISION": 0.9603424792846333,
    "ACCURACY": 0.9719562855709735,
    "RECALL": 0.9591541470093101,
    "SENT_BREAK_PRECISION": null,
    "SENT_BREAK_RECALL": null,
    "SENT_BREAK_F1": null,
    "SENT_BREAK_ACCURACY": null,
    "SENT_BREAK_EXACT_MATCH": null
  },
  {
    "WORKBOOK_NAME": "BiGRU_CRF_RoBERTa.xlsx",
    "NN_LAYER1": "BiGRU+CRF",
    "TRANSFORMER_LAYER2": "RoBERTa",
    "SPLIT": "TEST",
    "BLEU": 0.8194273778778034,
    "ROUGE_1_F1": 0.9225942801386773,
    "ROUGE_2_F1": 0.852322055620903,
    "ROUGE_L_F1": 0.8801978505678816,
    "METEOR": 0.9008846235919223,
    "BERTSCORE_P": 0.9834074480317816,
    "BERTSCORE_R": 0.9866347660985049,
    "BERTSCORE_F1": 0.9849926919654298,
    "F1": 0.9644028374463933,
    "PRECISION": 0.9661849295229392,
    "ACCURACY": 0.9758535339627262,
    "RECALL": 0.9653354291712203,
    "SENT_BREAK_PRECISION": 0.7868512110726643,
    "SENT_BREAK_RECALL": 0.7682432432432432,
    "SENT_BREAK_F1": 0.7774358974358975,
    "SENT_BREAK_ACCURACY": 0.970393614844123,
    "SENT_BREAK_EXACT_MATCH": 0.8225668996079768
  },
  {
    "WORKBOOK_NAME": "BiLSTM_CRF_BERT.xlsx",
    "NN_LAYER1": "BiLSTM+CRF",
    "TRANSFORMER_LAYER2": "BERT",
    "SPLIT": "TRAIN",
    "BLEU": 0.8419524533814129,
    "ROUGE_1_F1": 0.9314465196223717,
    "ROUGE_2_F1": 0.8704419844733189,
    "ROUGE_L_F1": 0.8795436095184455,
    "METEOR": 0.9145679827392708,
    "BERTSCORE_P": 0.9853208931308937,
    "BERTSCORE_R": 0.9881836520257812,
    "BERTSCORE_F1": 0.9867291921756419,
    "F1": 0.9681304808934765,
    "PRECISION": 0.9684442824220812,
    "ACCURACY": 0.9789167746438241,
    "RECALL": 0.9704005233841505,
    "SENT_BREAK_PRECISION": null,
    "SENT_BREAK_RECALL": null,
    "SENT_BREAK_F1": null,
    "SENT_BREAK_ACCURACY": null,
    "SENT_BREAK_EXACT_MATCH": null
  },
  {
    "WORKBOOK_NAME": "BiLSTM_CRF_BERT.xlsx",
    "NN_LAYER1": "BiLSTM+CRF",
    "TRANSFORMER_LAYER2": "BERT",
    "SPLIT": "VALIDATION",
    "BLEU": 0.7977472993667428,
    "ROUGE_1_F1": 0.9092537836868662,
    "ROUGE_2_F1": 0.8333911734757358,
    "ROUGE_L_F1": 0.8490502612526255,
    "METEOR": 0.8894344940473763,
    "BERTSCORE_P": 0.979899401091863,
    "BERTSCORE_R": 0.9838915371934669,
    "BERTSCORE_F1": 0.9818626660785728,
    "F1": 0.9563141971808742,
    "PRECISION": 0.9568512966385377,
    "ACCURACY": 0.9709703122363609,
    "RECALL": 0.9589376020080292,
    "SENT_BREAK_PRECISION": null,
    "SENT_BREAK_RECALL": null,
    "SENT_BREAK_F1": null,
    "SENT_BREAK_ACCURACY": null,
    "SENT_BREAK_EXACT_MATCH": null
  },
  {
    "WORKBOOK_NAME": "BiLSTM_CRF_BERT.xlsx",
    "NN_LAYER1": "BiLSTM+CRF",
    "TRANSFORMER_LAYER2": "BERT",
    "SPLIT": "TEST",
    "BLEU": 0.8157029759742813,
    "ROUGE_1_F1": 0.9183207530529598,
    "ROUGE_2_F1": 0.8486891459453669,
    "ROUGE_L_F1": 0.8607646515100212,
    "METEOR": 0.8996765171343359,
    "BERTSCORE_P": 0.9823657243971485,
    "BERTSCORE_R": 0.9859399470231268,
    "BERTSCORE_F1": 0.984122515974326,
    "F1": 0.9623045806626394,
    "PRECISION": 0.9629625832580848,
    "ACCURACY": 0.9747070871698807,
    "RECALL": 0.9647034333924467,
    "SENT_BREAK_PRECISION": 0.7181031238238615,
    "SENT_BREAK_RECALL": 0.6394101876675603,
    "SENT_BREAK_F1": 0.6764758021627371,
    "SENT_BREAK_ACCURACY": 0.9586505347108936,
    "SENT_BREAK_EXACT_MATCH": 0.7429691494801431
  },
  {
    "WORKBOOK_NAME": "BiLSTM_CRF_DeBERTa_v3.xlsx",
    "NN_LAYER1": "BiLSTM+CRF",
    "TRANSFORMER_LAYER2": "DeBERTa_v3",
    "SPLIT": "TRAIN",
    "BLEU": 0.8419524533814129,
    "ROUGE_1_F1": 0.9314465196223717,
    "ROUGE_2_F1": 0.8704419844733189,
    "ROUGE_L_F1": 0.8795436095184455,
    "METEOR": 0.9145679827392708,
    "BERTSCORE_P": 0.9853208931308937,
    "BERTSCORE_R": 0.9881836520257812,
    "BERTSCORE_F1": 0.9867291921756419,
    "F1": 0.9681304808934765,
    "PRECISION": 0.9684442824220812,
    "ACCURACY": 0.9789167746438241,
    "RECALL": 0.9704005233841505,
    "SENT_BREAK_PRECISION": null,
    "SENT_BREAK_RECALL": null,
    "SENT_BREAK_F1": null,
    "SENT_BREAK_ACCURACY": null,
    "SENT_BREAK_EXACT_MATCH": null
  },
  {
    "WORKBOOK_NAME": "BiLSTM_CRF_DeBERTa_v3.xlsx",
    "NN_LAYER1": "BiLSTM+CRF",
    "TRANSFORMER_LAYER2": "DeBERTa_v3",
    "SPLIT": "VALIDATION",
    "BLEU": 0.7977472993667428,
    "ROUGE_1_F1": 0.9092537836868662,
    "ROUGE_2_F1": 0.8333911734757358,
    "ROUGE_L_F1": 0.8490502612526255,
    "METEOR": 0.8894344940473763,
    "BERTSCORE_P": 0.979899401091863,
    "BERTSCORE_R": 0.9838915371934669,
    "BERTSCORE_F1": 0.9818626660785728,
    "F1": 0.9563141971808742,
    "PRECISION": 0.9568512966385377,
    "ACCURACY": 0.9709703122363609,
    "RECALL": 0.9589376020080292,
    "SENT_BREAK_PRECISION": null,
    "SENT_BREAK_RECALL": null,
    "SENT_BREAK_F1": null,
    "SENT_BREAK_ACCURACY": null,
    "SENT_BREAK_EXACT_MATCH": null
  },
  {
    "WORKBOOK_NAME": "BiLSTM_CRF_DeBERTa_v3.xlsx",
    "NN_LAYER1": "BiLSTM+CRF",
    "TRANSFORMER_LAYER2": "DeBERTa_v3",
    "SPLIT": "TEST",
    "BLEU": 0.8157029759742813,
    "ROUGE_1_F1": 0.9183207530529598,
    "ROUGE_2_F1": 0.8486891459453669,
    "ROUGE_L_F1": 0.8607646515100212,
    "METEOR": 0.8996765171343359,
    "BERTSCORE_P": 0.9823657243971485,
    "BERTSCORE_R": 0.9859399470231268,
    "BERTSCORE_F1": 0.984122515974326,
    "F1": 0.9623045806626394,
    "PRECISION": 0.9629625832580848,
    "ACCURACY": 0.9747070871698807,
    "RECALL": 0.9647034333924467,
    "SENT_BREAK_PRECISION": 0.8000688231245698,
    "SENT_BREAK_RECALL": 0.7791554959785523,
    "SENT_BREAK_F1": 0.7894736842105263,
    "SENT_BREAK_ACCURACY": 0.9719050208446619,
    "SENT_BREAK_EXACT_MATCH": 0.8321118118288734
  },
  {
    "WORKBOOK_NAME": "BiLSTM_CRF_ELECTRA.xlsx",
    "NN_LAYER1": "BiLSTM+CRF",
    "TRANSFORMER_LAYER2": "ELECTRA",
    "SPLIT": "TRAIN",
    "BLEU": 0.8419524533814129,
    "ROUGE_1_F1": 0.9314465196223717,
    "ROUGE_2_F1": 0.8704419844733189,
    "ROUGE_L_F1": 0.8795436095184455,
    "METEOR": 0.9145679827392708,
    "BERTSCORE_P": 0.9853208931308937,
    "BERTSCORE_R": 0.9881836520257812,
    "BERTSCORE_F1": 0.9867291921756419,
    "F1": 0.9681304808934765,
    "PRECISION": 0.9684442824220812,
    "ACCURACY": 0.9789167746438241,
    "RECALL": 0.9704005233841505,
    "SENT_BREAK_PRECISION": null,
    "SENT_BREAK_RECALL": null,
    "SENT_BREAK_F1": null,
    "SENT_BREAK_ACCURACY": null,
    "SENT_BREAK_EXACT_MATCH": null
  },
  {
    "WORKBOOK_NAME": "BiLSTM_CRF_ELECTRA.xlsx",
    "NN_LAYER1": "BiLSTM+CRF",
    "TRANSFORMER_LAYER2": "ELECTRA",
    "SPLIT": "VALIDATION",
    "BLEU": 0.7977472993667428,
    "ROUGE_1_F1": 0.9092537836868662,
    "ROUGE_2_F1": 0.8333911734757358,
    "ROUGE_L_F1": 0.8490502612526255,
    "METEOR": 0.8894344940473763,
    "BERTSCORE_P": 0.979899401091863,
    "BERTSCORE_R": 0.9838915371934669,
    "BERTSCORE_F1": 0.9818626660785728,
    "F1": 0.9563141971808742,
    "PRECISION": 0.9568512966385377,
    "ACCURACY": 0.9709703122363609,
    "RECALL": 0.9589376020080292,
    "SENT_BREAK_PRECISION": null,
    "SENT_BREAK_RECALL": null,
    "SENT_BREAK_F1": null,
    "SENT_BREAK_ACCURACY": null,
    "SENT_BREAK_EXACT_MATCH": null
  },
  {
    "WORKBOOK_NAME": "BiLSTM_CRF_ELECTRA.xlsx",
    "NN_LAYER1": "BiLSTM+CRF",
    "TRANSFORMER_LAYER2": "ELECTRA",
    "SPLIT": "TEST",
    "BLEU": 0.8157029759742813,
    "ROUGE_1_F1": 0.9183207530529598,
    "ROUGE_2_F1": 0.8486891459453669,
    "ROUGE_L_F1": 0.8607646515100212,
    "METEOR": 0.8996765171343359,
    "BERTSCORE_P": 0.9823657243971485,
    "BERTSCORE_R": 0.9859399470231268,
    "BERTSCORE_F1": 0.984122515974326,
    "F1": 0.9623045806626394,
    "PRECISION": 0.9629625832580848,
    "ACCURACY": 0.9747070871698807,
    "RECALL": 0.9647034333924467,
    "SENT_BREAK_PRECISION": 0.7118217750628818,
    "SENT_BREAK_RECALL": 0.6638739946380697,
    "SENT_BREAK_F1": 0.6870123114270852,
    "SENT_BREAK_ACCURACY": 0.9591036795359797,
    "SENT_BREAK_EXACT_MATCH": 0.7482529401738538
  },
  {
    "WORKBOOK_NAME": "BiLSTM_CRF_RoBERTa.xlsx",
    "NN_LAYER1": "BiLSTM+CRF",
    "TRANSFORMER_LAYER2": "RoBERTa",
    "SPLIT": "TRAIN",
    "BLEU": 0.8419524533814129,
    "ROUGE_1_F1": 0.9314465196223717,
    "ROUGE_2_F1": 0.8704419844733189,
    "ROUGE_L_F1": 0.8795436095184455,
    "METEOR": 0.9145679827392708,
    "BERTSCORE_P": 0.9853208931308937,
    "BERTSCORE_R": 0.9881836520257812,
    "BERTSCORE_F1": 0.9867291921756419,
    "F1": 0.9681304808934765,
    "PRECISION": 0.9684442824220812,
    "ACCURACY": 0.9789167746438241,
    "RECALL": 0.9704005233841505,
    "SENT_BREAK_PRECISION": null,
    "SENT_BREAK_RECALL": null,
    "SENT_BREAK_F1": null,
    "SENT_BREAK_ACCURACY": null,
    "SENT_BREAK_EXACT_MATCH": null
  },
  {
    "WORKBOOK_NAME": "BiLSTM_CRF_RoBERTa.xlsx",
    "NN_LAYER1": "BiLSTM+CRF",
    "TRANSFORMER_LAYER2": "RoBERTa",
    "SPLIT": "VALIDATION",
    "BLEU": 0.7977472993667428,
    "ROUGE_1_F1": 0.9092537836868662,
    "ROUGE_2_F1": 0.8333911734757358,
    "ROUGE_L_F1": 0.8490502612526255,
    "METEOR": 0.8894344940473763,
    "BERTSCORE_P": 0.979899401091863,
    "BERTSCORE_R": 0.9838915371934669,
    "BERTSCORE_F1": 0.9818626660785728,
    "F1": 0.9563141971808742,
    "PRECISION": 0.9568512966385377,
    "ACCURACY": 0.9709703122363609,
    "RECALL": 0.9589376020080292,
    "SENT_BREAK_PRECISION": null,
    "SENT_BREAK_RECALL": null,
    "SENT_BREAK_F1": null,
    "SENT_BREAK_ACCURACY": null,
    "SENT_BREAK_EXACT_MATCH": null
  },
  {
    "WORKBOOK_NAME": "BiLSTM_CRF_RoBERTa.xlsx",
    "NN_LAYER1": "BiLSTM+CRF",
    "TRANSFORMER_LAYER2": "RoBERTa",
    "SPLIT": "TEST",
    "BLEU": 0.8157029759742813,
    "ROUGE_1_F1": 0.9183207530529598,
    "ROUGE_2_F1": 0.8486891459453669,
    "ROUGE_L_F1": 0.8607646515100212,
    "METEOR": 0.8996765171343359,
    "BERTSCORE_P": 0.9823657243971485,
    "BERTSCORE_R": 0.9859399470231268,
    "BERTSCORE_F1": 0.984122515974326,
    "F1": 0.9623045806626394,
    "PRECISION": 0.9629625832580848,
    "ACCURACY": 0.9747070871698807,
    "RECALL": 0.9647034333924467,
    "SENT_BREAK_PRECISION": 0.7990213212163579,
    "SENT_BREAK_RECALL": 0.7660857908847185,
    "SENT_BREAK_F1": 0.782207014542344,
    "SENT_BREAK_ACCURACY": 0.9711573318832699,
    "SENT_BREAK_EXACT_MATCH": 0.824953127663201
  },
  {
    "WORKBOOK_NAME": "BiRNN_CRF_BERT.xlsx",
    "NN_LAYER1": "BiRNN+CRF",
    "TRANSFORMER_LAYER2": "BERT",
    "SPLIT": "TRAIN",
    "BLEU": 0.7503351421738268,
    "ROUGE_1_F1": 0.8827769717506064,
    "ROUGE_2_F1": 0.7969887873968022,
    "ROUGE_L_F1": 0.8343779478544239,
    "METEOR": 0.8567077056865432,
    "BERTSCORE_P": 0.9752650464466313,
    "BERTSCORE_R": 0.9801224360067753,
    "BERTSCORE_F1": 0.9776554313946539,
    "F1": 0.9458061724245517,
    "PRECISION": 0.951190265445923,
    "ACCURACY": 0.9631429706574263,
    "RECALL": 0.9448064819675609,
    "SENT_BREAK_PRECISION": null,
    "SENT_BREAK_RECALL": null,
    "SENT_BREAK_F1": null,
    "SENT_BREAK_ACCURACY": null,
    "SENT_BREAK_EXACT_MATCH": null
  },
  {
    "WORKBOOK_NAME": "BiRNN_CRF_BERT.xlsx",
    "NN_LAYER1": "BiRNN+CRF",
    "TRANSFORMER_LAYER2": "BERT",
    "SPLIT": "VALIDATION",
    "BLEU": 0.7246122920949926,
    "ROUGE_1_F1": 0.8697304979231144,
    "ROUGE_2_F1": 0.7739641693451388,
    "ROUGE_L_F1": 0.8158144045259204,
    "METEOR": 0.8416174095224188,
    "BERTSCORE_P": 0.9717354624015013,
    "BERTSCORE_R": 0.9772529974398751,
    "BERTSCORE_F1": 0.9744493247689917,
    "F1": 0.9387669845744645,
    "PRECISION": 0.9440643680279561,
    "ACCURACY": 0.9581663286766937,
    "RECALL": 0.9381756234399519,
    "SENT_BREAK_PRECISION": null,
    "SENT_BREAK_RECALL": null,
    "SENT_BREAK_F1": null,
    "SENT_BREAK_ACCURACY": null,
    "SENT_BREAK_EXACT_MATCH": null
  },
  {
    "WORKBOOK_NAME": "BiRNN_CRF_BERT.xlsx",
    "NN_LAYER1": "BiRNN+CRF",
    "TRANSFORMER_LAYER2": "BERT",
    "SPLIT": "TEST",
    "BLEU": 0.7434755310902229,
    "ROUGE_1_F1": 0.880264219706695,
    "ROUGE_2_F1": 0.7897788042244318,
    "ROUGE_L_F1": 0.83104854022962,
    "METEOR": 0.85233983912857,
    "BERTSCORE_P": 0.9741476915125264,
    "BERTSCORE_R": 0.9794490274993691,
    "BERTSCORE_F1": 0.9767542186904161,
    "F1": 0.9451277010517632,
    "PRECISION": 0.9507327465516517,
    "ACCURACY": 0.9626598983864686,
    "RECALL": 0.9438504131278715,
    "SENT_BREAK_PRECISION": 0.6265844467283316,
    "SENT_BREAK_RECALL": 0.6368384401114207,
    "SENT_BREAK_F1": 0.631669832498705,
    "SENT_BREAK_ACCURACY": 0.950667252584592,
    "SENT_BREAK_EXACT_MATCH": 0.7056417249019942
  },
  {
    "WORKBOOK_NAME": "BiRNN_CRF_DeBERTa_v3.xlsx",
    "NN_LAYER1": "BiRNN+CRF",
    "TRANSFORMER_LAYER2": "DeBERTa_v3",
    "SPLIT": "TRAIN",
    "BLEU": 0.7503351421738268,
    "ROUGE_1_F1": 0.8827769717506064,
    "ROUGE_2_F1": 0.7969887873968022,
    "ROUGE_L_F1": 0.8343779478544239,
    "METEOR": 0.8567077056865432,
    "BERTSCORE_P": 0.9752650464466313,
    "BERTSCORE_R": 0.9801224360067753,
    "BERTSCORE_F1": 0.9776554313946539,
    "F1": 0.9458061724245517,
    "PRECISION": 0.951190265445923,
    "ACCURACY": 0.9631429706574263,
    "RECALL": 0.9448064819675609,
    "SENT_BREAK_PRECISION": null,
    "SENT_BREAK_RECALL": null,
    "SENT_BREAK_F1": null,
    "SENT_BREAK_ACCURACY": null,
    "SENT_BREAK_EXACT_MATCH": null
  },
  {
    "WORKBOOK_NAME": "BiRNN_CRF_DeBERTa_v3.xlsx",
    "NN_LAYER1": "BiRNN+CRF",
    "TRANSFORMER_LAYER2": "DeBERTa_v3",
    "SPLIT": "VALIDATION",
    "BLEU": 0.7246122920949926,
    "ROUGE_1_F1": 0.8697304979231144,
    "ROUGE_2_F1": 0.7739641693451388,
    "ROUGE_L_F1": 0.8158144045259204,
    "METEOR": 0.8416174095224188,
    "BERTSCORE_P": 0.9717354624015013,
    "BERTSCORE_R": 0.9772529974398751,
    "BERTSCORE_F1": 0.9744493247689917,
    "F1": 0.9387669845744645,
    "PRECISION": 0.9440643680279561,
    "ACCURACY": 0.9581663286766937,
    "RECALL": 0.9381756234399519,
    "SENT_BREAK_PRECISION": null,
    "SENT_BREAK_RECALL": null,
    "SENT_BREAK_F1": null,
    "SENT_BREAK_ACCURACY": null,
    "SENT_BREAK_EXACT_MATCH": null
  },
  {
    "WORKBOOK_NAME": "BiRNN_CRF_DeBERTa_v3.xlsx",
    "NN_LAYER1": "BiRNN+CRF",
    "TRANSFORMER_LAYER2": "DeBERTa_v3",
    "SPLIT": "TEST",
    "BLEU": 0.7434755310902229,
    "ROUGE_1_F1": 0.880264219706695,
    "ROUGE_2_F1": 0.7897788042244318,
    "ROUGE_L_F1": 0.83104854022962,
    "METEOR": 0.85233983912857,
    "BERTSCORE_P": 0.9741476915125264,
    "BERTSCORE_R": 0.9794490274993691,
    "BERTSCORE_F1": 0.9767542186904161,
    "F1": 0.9451277010517632,
    "PRECISION": 0.9507327465516517,
    "ACCURACY": 0.9626598983864686,
    "RECALL": 0.9438504131278715,
    "SENT_BREAK_PRECISION": 0.7398345968297726,
    "SENT_BREAK_RECALL": 0.7475626740947076,
    "SENT_BREAK_F1": 0.7436785590578455,
    "SENT_BREAK_ACCURACY": 0.9657700580521312,
    "SENT_BREAK_EXACT_MATCH": 0.8009204022498722
  },
  {
    "WORKBOOK_NAME": "BiRNN_CRF_ELECTRA.xlsx",
    "NN_LAYER1": "BiRNN+CRF",
    "TRANSFORMER_LAYER2": "ELECTRA",
    "SPLIT": "TRAIN",
    "BLEU": 0.7503351421738268,
    "ROUGE_1_F1": 0.8827769717506064,
    "ROUGE_2_F1": 0.7969887873968022,
    "ROUGE_L_F1": 0.8343779478544239,
    "METEOR": 0.8567077056865432,
    "BERTSCORE_P": 0.9752650464466313,
    "BERTSCORE_R": 0.9801224360067753,
    "BERTSCORE_F1": 0.9776554313946539,
    "F1": 0.9458061724245517,
    "PRECISION": 0.951190265445923,
    "ACCURACY": 0.9631429706574263,
    "RECALL": 0.9448064819675609,
    "SENT_BREAK_PRECISION": null,
    "SENT_BREAK_RECALL": null,
    "SENT_BREAK_F1": null,
    "SENT_BREAK_ACCURACY": null,
    "SENT_BREAK_EXACT_MATCH": null
  },
  {
    "WORKBOOK_NAME": "BiRNN_CRF_ELECTRA.xlsx",
    "NN_LAYER1": "BiRNN+CRF",
    "TRANSFORMER_LAYER2": "ELECTRA",
    "SPLIT": "VALIDATION",
    "BLEU": 0.7246122920949926,
    "ROUGE_1_F1": 0.8697304979231144,
    "ROUGE_2_F1": 0.7739641693451388,
    "ROUGE_L_F1": 0.8158144045259204,
    "METEOR": 0.8416174095224188,
    "BERTSCORE_P": 0.9717354624015013,
    "BERTSCORE_R": 0.9772529974398751,
    "BERTSCORE_F1": 0.9744493247689917,
    "F1": 0.9387669845744645,
    "PRECISION": 0.9440643680279561,
    "ACCURACY": 0.9581663286766937,
    "RECALL": 0.9381756234399519,
    "SENT_BREAK_PRECISION": null,
    "SENT_BREAK_RECALL": null,
    "SENT_BREAK_F1": null,
    "SENT_BREAK_ACCURACY": null,
    "SENT_BREAK_EXACT_MATCH": null
  },
  {
    "WORKBOOK_NAME": "BiRNN_CRF_ELECTRA.xlsx",
    "NN_LAYER1": "BiRNN+CRF",
    "TRANSFORMER_LAYER2": "ELECTRA",
    "SPLIT": "TEST",
    "BLEU": 0.7434755310902229,
    "ROUGE_1_F1": 0.880264219706695,
    "ROUGE_2_F1": 0.7897788042244318,
    "ROUGE_L_F1": 0.83104854022962,
    "METEOR": 0.85233983912857,
    "BERTSCORE_P": 0.9741476915125264,
    "BERTSCORE_R": 0.9794490274993691,
    "BERTSCORE_F1": 0.9767542186904161,
    "F1": 0.9451277010517632,
    "PRECISION": 0.9507327465516517,
    "ACCURACY": 0.9626598983864686,
    "RECALL": 0.9438504131278715,
    "SENT_BREAK_PRECISION": 0.6636658264314008,
    "SENT_BREAK_RECALL": 0.6417130919220055,
    "SENT_BREAK_F1": 0.652504868118251,
    "SENT_BREAK_ACCURACY": 0.954599070240766,
    "SENT_BREAK_EXACT_MATCH": 0.728481336287711
  },
  {
    "WORKBOOK_NAME": "BiRNN_CRF_RoBERTa.xlsx",
    "NN_LAYER1": "BiRNN+CRF",
    "TRANSFORMER_LAYER2": "RoBERTa",
    "SPLIT": "TRAIN",
    "BLEU": 0.7503351421738268,
    "ROUGE_1_F1": 0.8827769717506064,
    "ROUGE_2_F1": 0.7969887873968022,
    "ROUGE_L_F1": 0.8343779478544239,
    "METEOR": 0.8567077056865432,
    "BERTSCORE_P": 0.9752650464466313,
    "BERTSCORE_R": 0.9801224360067753,
    "BERTSCORE_F1": 0.9776554313946539,
    "F1": 0.9458061724245517,
    "PRECISION": 0.951190265445923,
    "ACCURACY": 0.9631429706574263,
    "RECALL": 0.9448064819675609,
    "SENT_BREAK_PRECISION": null,
    "SENT_BREAK_RECALL": null,
    "SENT_BREAK_F1": null,
    "SENT_BREAK_ACCURACY": null,
    "SENT_BREAK_EXACT_MATCH": null
  },
  {
    "WORKBOOK_NAME": "BiRNN_CRF_RoBERTa.xlsx",
    "NN_LAYER1": "BiRNN+CRF",
    "TRANSFORMER_LAYER2": "RoBERTa",
    "SPLIT": "VALIDATION",
    "BLEU": 0.7246122920949926,
    "ROUGE_1_F1": 0.8697304979231144,
    "ROUGE_2_F1": 0.7739641693451388,
    "ROUGE_L_F1": 0.8158144045259204,
    "METEOR": 0.8416174095224188,
    "BERTSCORE_P": 0.9717354624015013,
    "BERTSCORE_R": 0.9772529974398751,
    "BERTSCORE_F1": 0.9744493247689917,
    "F1": 0.9387669845744645,
    "PRECISION": 0.9440643680279561,
    "ACCURACY": 0.9581663286766937,
    "RECALL": 0.9381756234399519,
    "SENT_BREAK_PRECISION": null,
    "SENT_BREAK_RECALL": null,
    "SENT_BREAK_F1": null,
    "SENT_BREAK_ACCURACY": null,
    "SENT_BREAK_EXACT_MATCH": null
  },
  {
    "WORKBOOK_NAME": "BiRNN_CRF_RoBERTa.xlsx",
    "NN_LAYER1": "BiRNN+CRF",
    "TRANSFORMER_LAYER2": "RoBERTa",
    "SPLIT": "TEST",
    "BLEU": 0.7434755310902229,
    "ROUGE_1_F1": 0.880264219706695,
    "ROUGE_2_F1": 0.7897788042244318,
    "ROUGE_L_F1": 0.83104854022962,
    "METEOR": 0.85233983912857,
    "BERTSCORE_P": 0.9741476915125264,
    "BERTSCORE_R": 0.9794490274993691,
    "BERTSCORE_F1": 0.9767542186904161,
    "F1": 0.9451277010517632,
    "PRECISION": 0.9507327465516517,
    "ACCURACY": 0.9626598983864686,
    "RECALL": 0.9438504131278715,
    "SENT_BREAK_PRECISION": 0.768910741301059,
    "SENT_BREAK_RECALL": 0.7078690807799443,
    "SENT_BREAK_F1": 0.7371283538796228,
    "SENT_BREAK_ACCURACY": 0.9664639082267502,
    "SENT_BREAK_EXACT_MATCH": 0.8048406340548833
  }
]''')

# ── Full Execution Orchestrator (16 Pipelines) ────────────────────────────────
def run_all_pipelines(config: PipelineConfig):
    start_total_time = time.time()
    set_seed(RANDOM_SEED)
    
    # 1. Load Dataset Splits
    splits = load_split_data(config.excel_path, config)
    char_vocab = build_char_vocab(splits["train"])
    embedding_weights = create_embedding_weights(len(char_vocab), config.embedding_dim, RANDOM_SEED)
    data_loaders = create_data_loaders(splits, char_vocab, config)
    
    # Pre-populate summary records with completed models (BiLSTM, BiGRU, BiRNN)
    # Any models executed in this run will dynamically overwrite their entries!
    summary_records = [dict(r) for r in PRECOMPUTED_12_MODELS_SUMMARY]
    current_nn_displays = [NN_DISPLAY_NAMES[m] for m in config.nn_models]
    summary_records = [r for r in summary_records if r["NN_LAYER1"] not in current_nn_displays]
    
    print("\n" + "=" * 70, flush=True)
    print("STARTING SEQUENTIAL TWO-STAGE TRAINING & BENCHMARKING", flush=True)
    print(f"Layer 1 Target NNs        : {config.nn_models} ({config.nn_epochs} Epochs each)", flush=True)
    print(f"Layer 2 Target Transformers: {config.transformer_models} ({config.transformer_epochs} Epochs each)", flush=True)
    print(f"Pre-existing summary rows  : {len(summary_records)} rows loaded from completed models", flush=True)
    print("=" * 70, flush=True)
    
    pipeline_idx = 0
    total_pipelines = len(config.nn_models) * len(config.transformer_models)
    
    for nn_key in config.nn_models:
        nn_display = NN_DISPLAY_NAMES[nn_key]
        print(f"\n{'#' * 70}\n>>> [STAGE 1 - LAYER 1]: Initializing Architecture: {nn_display}\n{'#' * 70}", flush=True)
        
        # Instantiate Layer 1 Neural Network + CRF
        nn_model = build_nn_model(nn_key, embedding_weights, config).to(device)
        optimizer_nn = torch.optim.Adam(nn_model.parameters(), lr=config.nn_lr, weight_decay=config.weight_decay)
        
        # 1. Train Neural Network for ALL 8 epochs first!
        nn_model = fit_neural_network(
            model=nn_model,
            train_loader=data_loaders["train"],
            val_loader=data_loaders["val"],
            optimizer=optimizer_nn,
            device=device,
            epochs=config.nn_epochs,
            patience=config.patience,
            model_name=nn_display,
        )
        
        # 2. Generate Cascaded Sliding Window Predictions across Train, Val, Test using trained NN
        print(f"\n>>> [STAGE 1 - LAYER 1]: Generating Cascaded Predictions for {nn_display} ...", flush=True)
        split_predictions = {}
        for split_name in ("train", "val", "test"):
            df_pred = predict_linewise_cascade(
                nn_model,
                splits[split_name],
                char_vocab,
                device,
                config.window_size
            )
            df_metrics = enrich_split_with_metrics(df_pred, config.use_bertscore, config.bertscore_batch_size)
            split_predictions[split_name] = (df_pred, df_metrics)
            print(f"    Completed cascaded inference on {split_name.upper()} split ({len(df_pred)} lines).", flush=True)
            
        df_pred_train, df_train_m = split_predictions["train"]
        df_pred_val, df_val_m = split_predictions["val"]
        df_pred_test, df_test_m = split_predictions["test"]
        
        # 3. For each of the 4 Transformers: attach fresh non-fine-tuned transformer & fine-tune 8 epochs!
        for trans_key in config.transformer_models:
            pipeline_idx += 1
            hf_trans_name = TRANSFORMER_REGISTRY[trans_key]
            wb_filename = f"{nn_display.replace('+', '_')}_{trans_key}.xlsx"
            wb_path = config.output_dir / wb_filename
            
            print(f"\n[{pipeline_idx}/{total_pipelines}] [STAGE 2 - LAYER 2]: Attaching NN ({nn_display}) -> Non-Fine-Tuned Transformer ({trans_key})", flush=True)
            print(f"    HuggingFace Pretrained Base: {hf_trans_name}", flush=True)
            
            # Instantiate fresh non-fine-tuned Transformer
            trans_module = TransformerSentenceBreakModule(trans_key, hf_trans_name, device, num_gpus)
            
            # Prepare dataset from NN's predicted lines (Train split)
            print(f"    Preparing Transformer training dataset from {nn_display} predicted output ...", flush=True)
            train_samples = trans_module.prepare_dataset(df_pred_train, config.max_seq_length)
            
            # Fine-tune Transformer for all 8 epochs on NN output
            print(f"    Fine-tuning {trans_key} for {config.transformer_epochs} epochs on {nn_display} output ...", flush=True)
            trans_module.fine_tune(
                train_samples,
                epochs=config.transformer_epochs,
                lr=config.transformer_lr,
                batch_size=config.transformer_batch_size
            )
            
            # Predict Sentence Breaks on NN Test predictions
            print(f"    Predicting sentence breaks on {nn_display} Test output using fine-tuned {trans_key} ...", flush=True)
            df_sb_test = trans_module.predict_sentence_breaks(
                df_pred_test,
                batch_size=config.transformer_batch_size * 2,
                max_len=config.max_seq_length
            )
            sb_metrics = compute_sentence_break_quality_metrics(df_sb_test)
            print(f"    [{nn_display} + {trans_key}] Sentence Break F1: {sb_metrics['SENT_BREAK_F1']:.4f} | Accuracy: {sb_metrics['SENT_BREAK_ACCURACY']:.4f}", flush=True)
            
            # Write Individual Workbook
            write_pipeline_workbook(
                wb_path,
                df_train_m,
                df_val_m,
                df_test_m,
                df_sb_test
            )
            
            # Collect Summary Rows for Master Summary Workbook
            for s_name, df_m in [("train", df_train_m), ("val", df_val_m), ("test", df_test_m)]:
                summary_records.append({
                    "WORKBOOK_NAME": wb_filename,
                    "NN_LAYER1": nn_display,
                    "TRANSFORMER_LAYER2": trans_key,
                    "SPLIT": s_name.upper(),
                    "BLEU": df_m["BLEU"].mean(),
                    "ROUGE_1_F1": df_m["ROUGE_1_F1"].mean(),
                    "ROUGE_2_F1": df_m["ROUGE_2_F1"].mean(),
                    "ROUGE_L_F1": df_m["ROUGE_L_F1"].mean(),
                    "METEOR": df_m["METEOR"].mean(),
                    "BERTSCORE_P": df_m["BERTSCORE_P"].mean(),
                    "BERTSCORE_R": df_m["BERTSCORE_R"].mean(),
                    "BERTSCORE_F1": df_m["BERTSCORE_F1"].mean(),
                    "F1": df_m["F1"].mean(),
                    "PRECISION": df_m["PRECISION"].mean(),
                    "ACCURACY": df_m["ACCURACY"].mean(),
                    "RECALL": df_m["RECALL"].mean(),
                    "SENT_BREAK_PRECISION": sb_metrics["SENT_BREAK_PRECISION"] if s_name == "test" else float("nan"),
                    "SENT_BREAK_RECALL": sb_metrics["SENT_BREAK_RECALL"] if s_name == "test" else float("nan"),
                    "SENT_BREAK_F1": sb_metrics["SENT_BREAK_F1"] if s_name == "test" else float("nan"),
                    "SENT_BREAK_ACCURACY": sb_metrics["SENT_BREAK_ACCURACY"] if s_name == "test" else float("nan"),
                    "SENT_BREAK_EXACT_MATCH": sb_metrics["SENT_BREAK_EXACT_MATCH"] if s_name == "test" else float("nan"),
                })
                
            # Clean up memory
            del trans_module
            torch.cuda.empty_cache()
            gc.collect()
            
        del nn_model
        torch.cuda.empty_cache()
        gc.collect()
        
    # Write Master Summary Workbook
    summary_wb_path = config.output_dir / "All_16_Models_Summary.xlsx"
    write_master_summary_workbook(summary_wb_path, summary_records)
    
    elapsed = time.time() - start_total_time
    print("\n" + "=" * 70, flush=True)
    print("ALL TARGET WORKBOOKS AND MASTER SUMMARY SUCCESSFULLY CREATED!", flush=True)
    print(f"Total Execution Time: {elapsed/60:.2f} minutes", flush=True)
    print(f"Output files stored at: {config.output_dir.resolve()}", flush=True)
    print("=" * 70, flush=True)

# Run the full pipeline
if __name__ == "__main__":
    run_all_pipelines(config)


Loading dataset from: /kaggle/input/datasets/vallurikeerthiram/scriptio-continua-eng-50-char/2000_articles_50_character_dataset_ART_ID_split_70_15_15.xlsx ...
  TRAIN     : 29,155 valid rows across 1,400 unique articles.
  VAL       : 6,584 valid rows across 300 unique articles.
  TEST      : 5,867 valid rows across 300 unique articles.
Character Vocabulary size: 65 unique characters.

STARTING SEQUENTIAL TWO-STAGE TRAINING & BENCHMARKING
Layer 1 Target NNs        : ['cnn'] (8 Epochs each)
Layer 2 Target Transformers: ['DeBERTa_v3', 'RoBERTa', 'BERT', 'ELECTRA'] (8 Epochs each)
Pre-existing summary rows  : 36 rows loaded from completed models

######################################################################
>>> [STAGE 1 - LAYER 1]: Initializing Architecture: CNN+CRF
######################################################################

--- Training Layer 1 NN [CNN+CRF] for 8 epochs (Early stopping patience=2) ---
  [NN: CNN+CRF] Epoch 1/8 | Train Loss: 8.9162 | Val Loss: 6.1703 

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

... (42 outputs left out)

[transformers] ElectraForTokenClassification LOAD REPORT from: google/electra-base-discriminator
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense.bias              | UNEXPECTED | 
electra.embeddings_project.weight                 | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
electra.embeddings_project.bias                   | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
classifier.weight                                 | MISSING    | 
classifier.bias                                   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream t

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

    Preparing Transformer training dataset from CNN+CRF predicted output ...
    Fine-tuning ELECTRA for 8 epochs on CNN+CRF output ...
    [Transformer: ELECTRA] Epoch 1/8 Loss: 0.2173
    [Transformer: ELECTRA] Epoch 2/8 Loss: 0.1366
    [Transformer: ELECTRA] Epoch 3/8 Loss: 0.0928
    [Transformer: ELECTRA] Epoch 4/8 Loss: 0.0636
    [Transformer: ELECTRA] Epoch 5/8 Loss: 0.0461
    [Transformer: ELECTRA] Epoch 6/8 Loss: 0.0342
    [Transformer: ELECTRA] Epoch 7/8 Loss: 0.0280
    [Transformer: ELECTRA] Epoch 8/8 Loss: 0.0238
    Predicting sentence breaks on CNN+CRF Test output using fine-tuned ELECTRA ...
    [CNN+CRF + ELECTRA] Sentence Break F1: 0.7057 | Accuracy: 0.9615
Writing Excel Workbook: CNN_CRF_ELECTRA.xlsx ...

Writing Master Summary Workbook: All_16_Models_Summary.xlsx ...

ALL TARGET WORKBOOKS AND MASTER SUMMARY SUCCESSFULLY CREATED!
Total Execution Time: 231.75 minutes
Output files stored at: /kaggle/working
